In [ ]:
#With importlib.reload(module) I can update my utils and do not restart the kernel every time
import importlib
# import my_utils.detector_v4 as ud
# import my_utils.data_analysis_v4 as uda
# import my_utils.plt_style as ps

import h5py
import hdf5plugin

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import cv2

from tqdm import tqdm   #progressbar

from scipy.optimize import curve_fit
import matplotlib.ticker as mticker
from matplotlib.animation import FuncAnimation
from IPython.display import display, Image as IPImage

# ps.article_style()

### Define and import all the data, it creates also a folder to save all the datasets

In [ ]:
day = '' 
scan_name = ''

f_name = f'/{day}/{scan_name}'
print(f"Percorso: {f_name}")

all_entries = next(os.walk(f_name))[2]
im_names = sorted([f for f in all_entries if f.endswith('.h5') and not f.startswith('.')])
# Cartella dove salvare i dataset
ds_folder = os.path.join(f_name, 'ds')

if not os.path.exists(ds_folder):
    os.makedirs(ds_folder)
    print(f"✅ Cartella creata: {ds_folder}")
else:
    print(f"📁 La cartella esiste già: {ds_folder}")

#folder to save plots
plot_folder = f'/{day}/{scan_name}'
if not os.path.exists(plot_folder):
    os.makedirs(plot_folder)

# Crea i percorsi completi per i file
file_names = [os.path.join(f_name, im_n) for im_n in im_names]
ds_folder = os.path.join(f_name, 'ds')

print(f"File trovati: {len(file_names)}")
if len(file_names) > 0:
    print(f"Primo file valido: {file_names[0]}")

In [ ]:
def hdf5_to_array(fileName, roi_L = 512, roi_x0 = 0, roi_y0 = 0):
    '''
    Read the hdf5 file and return the image as a numpy array of
    
    Params:
    fileName: str, path to the hdf5 file
    roi_L: int, length of the region of interest (ROI) in pixels
    roi_x0: int, x coordinate of the top left corner of the ROI
    roi_y0: int, y coordinate of the top left corner of the ROI   
    Returns:
    im: numpy array, the image data in the ROI
    '''
    
        
    with h5py.File(fileName, 'r') as f:
        dset = f['entry']['data']['data']   #this is how the h5 file is structured
        
        im = np.array(dset[:, roi_y0:(roi_y0+roi_L), roi_x0:(roi_x0+roi_L)], dtype=np.int32)
        
    return im

In [ ]:
import plotly.express as px
import nbformat

check_im = hdf5_to_array(file_names[0]).sum(0)
counts_ref_frame = check_im.sum()


fig = px.imshow(check_im, color_continuous_scale='Viridis', title="Use to set center and roi")
fig.show()

In [ ]:
rough_xc, rough_yc = 283, 282  ##CHANGE ME
roi_L = 512
roi_x0, roi_y0 = rough_xc-roi_L//2, rough_yc-roi_L//2

#plot the image with the ROI and print tot sum count
#check different files and use this to set the threshold
check_frame = hdf5_to_array(fileName = file_names[-10], roi_L=roi_L, roi_x0=roi_x0, roi_y0=roi_y0)[0,:,:]
plt.imshow(check_frame)
plt.title('Image with ROI')
plt.colorbar()

check_frame_sum = check_frame.sum()
print('Sum counts in the ROI: ', check_frame_sum)

########CHECK ME##############
threshold = check_frame_sum * 10
print('Assigned threshold: ', threshold)

In [ ]:
def align_images(image, reference_image, number_of_iterations = 1000, termination_eps = 1e-9):
    '''
    Aligns the input image to the reference image using the Enhanced Correlation Coefficient (ECC) algorithm.
    The ECC algorithm maximizes the correlation coefficient between the input image and the reference image
    by estimating a transformation matrix iteratively. This method is robust and widely used for image alignment tasks.
    
    The function uses a translation motion model to align the images. It initializes a 2x3 transformation matrix
    to the identity matrix and iteratively updates it to maximize the correlation coefficient. The aligned image
    is obtained by applying the inverse of the transformation matrix to the input image.
    
    Reference:
    Evangelidis, G. D., & Psarakis, E. Z. (2008). Parametric Image Alignment using Enhanced Correlation Coefficient (ECC) Maximization.
    IEEE Transactions on Pattern Analysis and Machine Intelligence, 30(10), 1858–1865. https://doi.org/10.1109/TPAMI.2008.113
    
    Params:
    image: np.ndarray, the input image to be aligned
    reference_image: np.ndarray, the reference image to align to
    number_of_iterations: int, the maximum number of iterations for the ECC algorithm (default: 10)
    termination_eps: float, the threshold for the increment in the correlation coefficient between iterations (default: 1e-3)
    
    Returns:
    aligned_image: np.ndarray, the aligned image
    '''
    
    # Define the motion model
    warp_mode = cv2.MOTION_TRANSLATION
    
    # Define 2x3 transformation matrix and initialize to identity
    warp_matrix = np.eye(2, 3, dtype=np.float32) #float necessary for algorithm
    
    
    # Define termination criteria
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, number_of_iterations, termination_eps)
    
    # Run the ECC algorithm to find the warp matrix
    cc, warp_matrix = cv2.findTransformECC(reference_image.astype(np.float32), image.astype(np.float32), warp_matrix, warp_mode, criteria)
    
    # Use warpAffine for Translation, Euclidean and Affine
    aligned_image = cv2.warpAffine(image.astype(np.float32), warp_matrix, 
                                   (reference_image.astype(np.float32).shape[1], reference_image.astype(np.float32).shape[0]),
                                   flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP)
    
    return aligned_image

### Test

In [ ]:
ref_frame = check_frame.copy()
counts_ref_frame = ref_frame.sum()

#you can change the number in [] to check different frames
to_be_aligned_frame = hdf5_to_array(file_names[-10], roi_L, roi_x0, roi_y0)[-1,:,:]
# # Normalize the image to the same number of counts as the reference frame
# to_be_aligned_frame = to_be_aligned_frame/to_be_aligned_frame.sum() * counts_ref_frame

test_aligned_frame = align_images(to_be_aligned_frame, ref_frame)


#####Comparison plot
fig, axs = plt.subplots(2,2, dpi=300)
vmax = 90  ##change me
val = 50  ##change me

im = axs[0,0].imshow(ref_frame, vmax = vmax)
axs[0,0].set_title('Reference frame')
#colorbar
cbar = plt.colorbar(im, ax=axs[0,0])

axs[0,1].imshow(to_be_aligned_frame, vmax = vmax)
axs[0,1].set_title('Frame to be aligned')


#difference between the two images
im2 = axs[1,0].imshow(ref_frame - to_be_aligned_frame, cmap='seismic', vmin=-val, vmax=val)
axs[1,0].set_title('Original difference')
#colorbar
cbar = plt.colorbar(im2, ax=axs[1,0])

axs[1,1].imshow(ref_frame - test_aligned_frame, cmap='seismic', vmin=-val, vmax=val)
axs[1,1].set_title('Aligned difference')

plt.tight_layout()

## Alignement procedure (no normalization)

In [ ]:
def get_image(file_name, ref_frame):
    '''
    Reads the hdf5 file, aligns all frames that have a number of counts below the threshold and sums them to create the final image.
    The frames that are above the threshold are discarded.
    The function uses the Enhanced Correlation Coefficient (ECC) algorithm to align the images.
    The frames for which the alignment fails are discarded.
    The final image is normalized to account for the discarded frames.
    
    Params:
    file_name: str, path to the hdf5 file
    ref_frame: np.ndarray, the reference frame to align to
    Returns:
    im: np.ndarray, the aligned image data
    counts: list, the number of counts in each frame
    '''
    
    # Read the hdf5 file and get the image data
    frames = hdf5_to_array(file_name, roi_L, roi_x0, roi_y0)  # (N_frames, L, L)
    
    # The final image
    im = np.zeros_like(frames[0], dtype=np.float64)
    # Store the counts for sanity check
    counts = []
    # Align the frames to the reference frame
    for i, frame in enumerate(frames):
        if frame.sum() < threshold:
            try:
                # Align the image to the reference frame
                aligned_frame = align_images(frame, ref_frame)
                
                # Sum the aligned frames
                im += aligned_frame
                
                # Update counts
                counts.append(frame.sum())

            except cv2.error as e:
                print(f"ECC alignment failed: frame {i} discarded\nFile: {file_name}")
                # Plot the image
                plt.imshow(frame, vmax=val)
                plt.title('Bad frame')
                plt.show()
                
                # Update counts
                counts.append(0)
        else:
            print(f'Frame {i} above threshold discarded. Frame counts = {frame.sum()}\nFile: {file_name}')
            # Update counts
            counts.append(0)
    
    # Normalize the final image to account for discarded frames
    if len(counts) > 0:
        im *= len(frames) / len([c for c in counts if c > 0])
    
    return im, counts


In [ ]:
#check function
im, counts = get_image(file_names[0], ref_frame)

fig, axs = plt.subplots(1, 2)

# Plot the aligned image
axs[0].imshow(im, cmap='viridis')
axs[0].set_title('Aligned Image')
axs[0].axis('off')

# Plot the counts
axs[1].plot(counts, marker='o', linestyle='-')
axs[1].set_title('Sum of Aligned Frames')
axs[1].set_xlabel('Frame Index')
axs[1].set_ylabel('Counts')

plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
from tqdm import tqdm

# 1. Definiamo i percorsi in modo corretto per macOS
data_path = os.path.join(ds_folder, 'aligned_images_no_norm.npy')
counts_path = os.path.join(ds_folder, 'counts.npy')

# 2. CONTROLLO: Se i file esistono già, li carichiamo e saltiamo il lavoro lungo
if os.path.exists(data_path) and os.path.exists(counts_path):
    print("🚀 File .npy trovati! Caricamento in corso (salto il loop)...")
    data = np.load(data_path)
    counts = np.load(counts_path)
    print(f"✅ Dati caricati correttamente. Shape: {data.shape}")
else:
    print("⏳ Dati non trovati. Inizio elaborazione immagini (richiederà tempo)...")
    data = []
    counts = []

    # loop over all files
    for fn in tqdm(file_names):
        # Assicurati che 'ref_frame' sia stato definito correttamente prima
        im, count = get_image(fn, ref_frame)
        data.append(im)
        counts.append(count)
        
    # Conversione in array
    data = np.array(data)
    counts = np.array(counts)

    # 3. SALVATAGGIO: Ora salviamo i file per la prossima volta
    np.save(data_path, data)
    np.save(counts_path, counts)
    print(f" Dati salvati con successo in: {ds_folder}")

In [ ]:
# open the data
data = np.load(os.path.join(ds_folder, 'aligned_images_no_norm.npy'))
counts_per_frame = np.load(os.path.join(ds_folder, 'counts.npy'))

# sanity check plot of counts per frame
fig = px.line(y=counts_per_frame.flatten(), title='Counts for all frames, sanity check')
fig.update_layout(xaxis_title='Frame Index', yaxis_title='Counts')
fig.show()


## Dataset creation

In [ ]:
def get_coordinates(im_names):
    '''
    Get the coordinates of the images from the file names.

    This function extracts metadata from the image file names, including lab time, fast scan index, delay time, and theta.
    The file names are expected to follow a specific format where the metadata is encoded as underscores-separated values.

    Params:
    im_names: list of str, the list of image file names

    Returns:
    lab_time: np.ndarray, array of datetime64[us] representing the lab time for each image
    fast_scans: np.ndarray, array of integers representing the fast scan index for each image
    delay_time: np.ndarray, array of floats representing the delay time for each image (rounded to the nearest 1000 femtoseconds)
    '''
    from datetime import datetime

    lab_time = np.zeros(len(im_names), dtype='datetime64[us]')
    fast_scans = np.zeros(len(im_names))
    delay_time = np.zeros(len(im_names))

    for i in range(len(im_names)):
        info = im_names[i].split('_')

        lab_time[i] = datetime(2025, *list(map(int, info[0:5])))
        fast_scans[i] = int(info[6])
        delay_time[i] = np.round(float(info[7][:-1]), -3)  # femtoseconds


    return lab_time, fast_scans, delay_time

# 1. Recuperiamo la lista dei file ignorando i file nascosti del Mac
# Filtriamo: deve finire con .h5 E non deve iniziare con un punto
im_names = [f for f in os.listdir(f_name) if f.endswith('.h5') and not f.startswith('.')]

# 2. Ordiniamo i file alfabeticamente per garantire la corretta sequenza temporale
im_names.sort()

# 3. Ora chiamiamo la funzione get_coordinates sulla lista pulita
lab_time, scan, delay_time = get_coordinates(im_names)

print(f"✅ Successo! Sono stati trovati ed elaborati {len(im_names)} file.")
print(f"Primo file elaborato: {im_names[0]}")

In [ ]:
# 1. Prendi le dimensioni REALI dall'array data
n_frames, ny, nx = data.shape

ds = xr.Dataset(
    data_vars=dict(
        im_old=(["lab_time", "y", "x"], data),
    ),

    coords=dict(
        x=(["x"], np.arange(nx)),
        y=(["y"], np.arange(ny)),
        lab_time=(["lab_time"], lab_time),
        # Usiamo 'scan' perché è così che l'hai chiamata nella Cella 11
        scan=(["lab_time"], scan), 
        delay_time=(["lab_time"], delay_time),
    )
)

# 2. Conversione tempi
ds['delay_time'] = ds.delay_time * 1e-3 

# 3. Salvataggio sicuro per Mac
import os
save_path = os.path.join(ds_folder, "ds_no_norm.nc")
ds.to_netcdf(save_path)

print(f"✅ Dataset salvato! Dimensioni: {nx}x{ny} pixel, {n_frames} immagini.")

# Costruiamo il percorso in modo pulito per Mac
file_path = os.path.join(ds_folder, "ds_no_norm.nc")

ds

### Normalization (on counts)

In [ ]:
ds.im_old.sum(('x','y')).plot(label='sum counts')

#I don't like to loose the information about the number of counts in the image, so do this: * ds_us.im_no_norm.sum(('x','y')).mean()
ds['im'] = ds.im_old / ds.im_old.sum(('x','y')) * ds.im_old.sum(('x','y')).mean()

ds.im.sum(('x','y')).plot(label='norm counts')

### Order by delay time

In [ ]:
import numpy as np
import cv2
import xarray as xr
import scipy.ndimage as nd

def radial_average_python(img, cx, cy, n_bins=215):
    y, x = np.indices(img.shape)
    r = np.sqrt((x - cx)**2 + (y - cy)**2)
    r_int = r.astype(int)
    tbin = np.bincount(r_int.ravel(), img.ravel())
    nr = np.bincount(r_int.ravel())
    radial_profile = np.divide(tbin, nr, out=np.zeros_like(tbin, dtype=float), where=nr!=0)
    return radial_profile[:n_bins]


In [ ]:
#Unstack the data 
ds_us = ds.set_index(lab_time=('delay_time', 'scan'))

# 864/24 = 36
# This should be 0, otherwise the unstack will not work, which means you didn't put the right coordinates in the get_coordinates function
print('number of duplicates = ', ds_us.im.indexes['lab_time'].duplicated().sum())
ds_us = ds_us.drop_duplicates('lab_time')
ds_us = ds_us.unstack(('lab_time'))

ds_us

In [ ]:
# Calcoliamo il centro sulle immagini che hai appena finito di allineare in ds_us
ref_frame_mean = ds_us.im.mean(dim=['scan', 'delay_time']).values
cy_real, cx_real = nd.maximum_position(nd.gaussian_filter(ref_frame_mean, 3))

print(f"✅ Centro trovato: X={cx_real}, Y={cy_real}")

In [ ]:
import hvplot.xarray

vmax = 1000
vmin = 000
ds_us.im.isel(delay_time=5, scan=5).hvplot.image(x='x', y='y', clim=(vmin, vmax), cmap ='viridis', aspect='square',
                                       title='To be used for ROI selection')

In [ ]:
vmax = 600
vmin = 0

fig, ax = plt.subplots()
data = ds_us.im.isel(delay_time=0, scan=0).values
im = ax.imshow(data, vmax=vmax, vmin=vmin, cmap='viridis')
ax.set_title(f"Delay time: {int(ds_us.delay_time.isel(delay_time=0).item())} ps, Scan: {int(ds_us.scan.isel(scan=0).item())}")
plt.colorbar(im, ax=ax)
plt.show()

### Do the plot for all the radial profiles of all frames of a single image

In [ ]:
import plotly.graph_objects as go
import numpy as np

def get_radial_profile_single(img, center):
    """Calcola il profilo radiale di una singola matrice 2D."""
    y, x = np.indices(img.shape)
    # Calcolo della distanza euclidea dal centro
    r = np.sqrt((x - center[1])**2 + (y - center[0])**2).astype(int)
    
    # Somma dei conteggi per ogni raggio e conteggio dei pixel per ogni raggio
    tbin = np.bincount(r.ravel(), img.ravel())
    nr = np.bincount(r.ravel())
    
    # Calcolo della media evitando divisioni per zero
    radial_profile = np.divide(tbin, nr, out=np.zeros_like(tbin, dtype=float), where=nr!=0)
    return np.arange(len(radial_profile)), radial_profile

# 1. Selezioniamo il file (puoi cambiare l'indice [0] per vedere altri scan/delay)
fn_example = file_names[0]
frames_example = hdf5_to_array(fn_example, roi_L, roi_x0, roi_y0) # Carica i 20 frame

# 2. Definiamo il centro e il raggio massimo da visualizzare
center_roi = (roi_L // 2, roi_L // 2)
max_display_r = 215 

# 3. Creazione del grafico interattivo con Plotly
fig = go.Figure()

# Usiamo una scala cromatica per rendere il grafico leggibile
from plotly.colors import sample_colorscale
colors = sample_colorscale("Viridis", np.linspace(0, 1, 20))

for i in range(20):
    r, prof = get_radial_profile_single(frames_example[i], center_roi)
    
    # Filtriamo i dati fino a max_display_r
    mask = r < max_display_r
    
    fig.add_trace(go.Scatter(
        x=r[mask], 
        y=prof[mask],
        mode='lines',
        name=f'Frame {i}',
        line=dict(color=colors[i], width=1.5),
        opacity=0.7
    ))

# Configurazione del layout
fig.update_layout(
    title=f"Confronto Profili Radiali - 20 Frame del file: {os.path.basename(fn_example)}",
    xaxis_title="Raggio (pixel)",
    yaxis_title="Intensità Media (counts)",
    template="plotly_white",
    legend=dict(title="Frames", traceorder="normal"),
    xaxis=dict(range=[0, max_display_r]),
)

fig.show()

### Do the radial average of a single image

In [ ]:
import numpy as np
import plotly.graph_objects as go
import os

# 1. Carichiamo i 20 frame di un file specifico
fn = file_names[0] 
frames = hdf5_to_array(fn, roi_L, roi_x0, roi_y0)

# 2. Allineamento interno: usiamo il primo frame di QUESTO file come riferimento
# Questo assicura che non ci siano spostamenti "globali" rispetto al centro (256,256)
local_ref = frames[0].astype(np.float32)
aligned_frames = [local_ref] # Il primo è già allineato a se stesso

print(f"Allineamento dei 20 frame di {os.path.basename(fn)}...")
for i in range(1, 20):
    try:
        # Allineiamo il frame i-esimo al primo frame del file
        aligned = align_images(frames[i], local_ref)
        aligned_frames.append(aligned)
    except:
        # Se l'allineamento fallisce, teniamo il frame originale per non perdere dati
        aligned_frames.append(frames[i])

# 3. Calcolo dell'immagine media (ora che sono allineati)
mean_img = np.mean(aligned_frames, axis=0)

# 4. Funzione profilo radiale (assicuriamoci che usi float64 per la precisione)
def get_radial_profile_clean(img, center):
    y, x = np.indices(img.shape)
    r = np.sqrt((x - center[1])**2 + (y - center[0])**2).astype(int)
    tbin = np.bincount(r.ravel(), img.ravel())
    nr = np.bincount(r.ravel())
    # Divisione float
    profile = np.divide(tbin, nr, out=np.zeros(len(tbin), dtype=np.float64), where=nr!=0)
    return np.arange(len(profile)), profile

# 5. Calcolo e Plot Interattivo
center_roi = (roi_L // 2, roi_L // 2)
r, profile = get_radial_profile_clean(mean_img, center_roi)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=r, 
    y=profile, 
    mode='lines',
    name='Profilo Medio (Allineato)',
    line=dict(color='firebrick', width=2)
))

fig.update_layout(
    title=f"Profilo Radiale Medio (20 frame allineati) - Delay: {int(ds_us.delay_time.isel(delay_time=0).item())} ps - Scan: {int(ds_us.scan.isel(scan=0).item())}",
    xaxis_title="Raggio (pixel)",
    yaxis_title="Intensità Media (counts)",
    xaxis=dict(range=[0, 215]),
    template="plotly_white",
    hovermode="x"
)

fig.show()

### Compute the radial average for all the images

In [ ]:
# --- ESECUZIONE DEL CALCOLO SU TUTTO IL DATASET ---
radial_profiles = xr.apply_ufunc(
    radial_average_python,        # La funzione core
    ds_us.im,                    # Il dataset con tutte le immagini
    input_core_dims=[["y", "x"]], 
    output_core_dims=[["radius"]],
    # Qui passiamo i centri TROVATI AUTOMATICAMENTE (es. 258, 238)
    kwargs={'cx': cx_real, 'cy': cy_real, 'n_bins': 215}, 
    vectorize=True,              # Fa il loop su ogni scan e ogni delay_time
    output_dtypes=[float]
)

# Aggiungiamo la coordinata del raggio
radial_profiles = radial_profiles.assign_coords(radius=np.arange(215))

print("✅ Calcolo completato su tutti gli scan e delay!")

Il problema che c'era è che io avevo fissato il centro dove facevo la media radiale a 256,256 (il centro del quadrato), ma il problema è che l'ECC ALLINEA MA NON CENTRA, quindi il picco centrale è in un'altra posizione. Non ho fatto altro che modificare le coordinate del centro da cui calcolo la media, è tutto automatico, fa lui da solo.

# Fitting

## FIRST PEAK

### Fit the first peak

Double gaussian

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. DEFINIZIONE FUNZIONE ---
def double_gaussian_with_linear_bg(r, A1, mu1, sigma1, A2, mu2, sigma2, slope, intercept):
    g1 = A1 * np.exp(-(r - mu1)**2 / (2 * sigma1**2))
    g2 = A2 * np.exp(-(r - mu2)**2 / (2 * sigma2**2))
    return g1 + g2 + slope * r + intercept

# --- 2. SETTAGGI ---
r_min = 47
r_max = 71

# *** UNICO PARAMETRO DA CAMBIARE: il confine tra i due picchi ***
# Guardalo dal profilo radiale: dove c'è la valle tra i due picchi?
r_mid = 62  # <--- CAMBIA ME se necessario

delay_to_plot = 0

r_vals = radial_profiles['radius'].values
fit_mask = (r_vals >= r_min) & (r_vals <= r_max)
r_fit = r_vals[fit_mask]

n_delays = radial_profiles.sizes['delay_time']
n_scans  = radial_profiles.sizes['scan']

fit_results_double = np.zeros((n_delays, n_scans, 10))

# --- 3. FIGURA ---
ncols = 4
nrows = int(np.ceil(n_scans / ncols))
fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(n_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

# --- 4. FIT ---
print(f"Inizio il fit a doppia gaussiana su {n_scans} scan e {n_delays} delay...")

for i in range(n_scans):
    for j in range(n_delays):
        y_vals = radial_profiles.isel(delay_time=j, scan=i).values
        ydata  = y_vals[fit_mask]

        if not np.isfinite(ydata).all() or np.all(ydata == 0):
            fit_results_double[j, i, :] = np.nan
            continue

        A_guess = np.max(ydata) - np.min(ydata)
        p0 = [A_guess, r_min + (r_mid - r_min) * 0.5, 2.0,
              A_guess, r_mid + (r_max - r_mid) * 0.5, 2.0,
              0.0, np.min(ydata)]

        # Bounds che forzano mu1 < r_mid e mu2 > r_mid — nessuno swap necessario
        lower_bounds = [0, r_min, 0.1,   0, r_mid, 0.1,  -np.inf, -np.inf]
        upper_bounds = [np.inf, r_mid, r_mid-r_min,  np.inf, r_max, r_max-r_mid,  np.inf, np.inf]

        try:
            popt, _ = curve_fit(double_gaussian_with_linear_bg, r_fit, ydata,
                                p0=p0, bounds=(lower_bounds, upper_bounds), maxfev=3000)

            fit_results_double[j, i, 0] = popt[0]
            fit_results_double[j, i, 1] = popt[1]
            fit_results_double[j, i, 2] = popt[2]
            fit_results_double[j, i, 3] = popt[3]
            fit_results_double[j, i, 4] = popt[4]
            fit_results_double[j, i, 5] = popt[5]
            fit_results_double[j, i, 6] = popt[6]
            fit_results_double[j, i, 7] = popt[7]
            fit_results_double[j, i, 8] = popt[0] * popt[2]  # Area 1
            fit_results_double[j, i, 9] = popt[3] * popt[5]  # Area 2

            if j == delay_to_plot:
                y_fit_curve = double_gaussian_with_linear_bg(r_fit, *popt)
                row = (i // ncols) + 1
                col = (i % ncols) + 1
                fig.add_trace(go.Scatter(x=r_fit, y=ydata, mode='lines+markers',
                                         marker=dict(size=4), line=dict(width=1, color='royalblue'),
                                         showlegend=False), row=row, col=col)
                fig.add_trace(go.Scatter(x=r_fit, y=y_fit_curve, mode='lines',
                                         line=dict(dash='dash', width=2, color='red'),
                                         showlegend=False), row=row, col=col)

        except RuntimeError:
            fit_results_double[j, i, :] = np.nan

print("✅ Fit completato! Risultati salvati in 'fit_results_double'.")

# --- 5. PLOT ---
fig.update_layout(
    height=250 * nrows, width=300 * ncols,
    title=f"Verifica Fit Doppio per delay_idx={delay_to_plot} (Raggio {r_min}-{r_max}, split a r_mid={r_mid})",
    showlegend=False, hovermode="x unified", template="plotly_white"
)
fig.update_xaxes(title_text="Raggio (pixel)")
fig.update_yaxes(title_text="Intensità")
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# --- 1. SETTAGGI ---
n_cols = 4
# Prendiamo n_scans direttamente dal dataset se non è in memoria
n_scans = radial_profiles.sizes['scan']
n_rows = int(np.ceil(n_scans / n_cols))

# Estraiamo l'asse dei tempi
delay_times = radial_profiles['delay_time'].values

# --- 2. CREAZIONE FIGURA ---
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False # Lasciamo falso così ogni scan si adatta al proprio livello di segnale
)

# --- 3. PLOT DELLA DINAMICA PER OGNI SCAN ---
for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    # Estraiamo l'ampiezza (A) per tutti i delay dello scan 'j'
    # Ricorda: in fit_results l'indice 0 è l'Ampiezza (A)
    amplitudes = fit_results_double[:, j, 0]        

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes,
        mode='lines+markers',
        name=f"Scan {j}",
        marker=dict(size=5),
        line=dict(width=1.5),
        showlegend=False
    ), row=row, col=col)

# --- 4. LAYOUT FINALE ---
fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Dinamica dell'Ampiezza (A) vs Delay Time per ogni Scan",
    showlegend=False,
    hovermode="x unified"
)

fig.update_yaxes(title_text="Ampiezza (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# --- SETTAGGI PER QUESTO ESPERIMENTO ---
# ==========================================

# 1. Quale delay usiamo come riferimento "freddo"? (indice 0 = primo delay)
idx_baseline = 0 

# 2. Ci sono scan palesemente sballati che vuoi escludere dalla media finale?
bad_scans = [] 


parametro_scelto = 0

# ==========================================

# --- PREPARAZIONE DEI DATI ---
delays = radial_profiles['delay_time'].values
n_scans = radial_profiles.sizes['scan']

# Estraiamo l'intensità assoluta (Dati Puri)
dati_intensita = fit_results_double[:, :, parametro_scelto]

# Calcoliamo la normalizzazione (I / I_0) usando il delay scelto come baseline
intensita_t0 = dati_intensita[idx_baseline, :]
norm_intensita = dati_intensita / intensita_t0

# Identifichiamo gli scan buoni per fare la media
good_scans = [i for i in range(n_scans) if i not in bad_scans]

layout_template = dict(hovermode="x unified", xaxis_title="Delay Time", 
                       height=400, width=900, margin=dict(l=50, r=50, t=50, b=50))
# --- FIGURA 3: MEDIA FINALE (Solo scan buoni) ---
fig3 = go.Figure()

# 1. Estraiamo i dati puri degli scan buoni
dati_puri_good = dati_intensita[:, good_scans]

# 2. Facciamo la media dei dati PURI (cancella il rumore!)
mean_pura = np.nanmean(dati_puri_good, axis=1)

# 3. Normalizziamo la media usando i primi 3 punti (come fa il fit)
baseline_media = np.nanmean(mean_pura[:3])
mean_good_normalizzata = mean_pura / baseline_media

fig3.add_trace(go.Scatter(x=delays, y=mean_good_normalizzata, mode='lines+markers', 
                          line=dict(color='blue', width=3), 
                          name=f'Media Normalizzata ({len(good_scans)} scan)'))

fig3.update_layout(**layout_template, title=f"3. Media Intensità Reale (Esclusi scan: {bad_scans})", yaxis_title="Intensità Normalizzata (I / I₀)")
# Aggiungiamo la linea di riferimento all'1.0
fig3.add_hline(y=1.0, line_dash="dash", line_color="black")
fig3.show()

Single Gaussian

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. DEFINIZIONE FUNZIONE ---
def gaussian_with_linear_bg(r, A, mu, sigma, slope, intercept):
    return A * np.exp(-(r - mu)**2 / (2 * sigma**2)) + slope * r + intercept

# --- 2. SETTAGGI DEL RANGE E DEL DELAY DA PLOTTARE ---
r_min = 57   # Sostituisci con 140 se vuoi il picco 5 del tuo script MATLAB
r_max = 88   # Sostituisci con 155 se vuoi il picco 5 del tuo script MATLAB

delay_to_plot = 0  # L'indice del delay che vuoi ispezionare visivamente

# Estrazione asse x (raggio) e maschera
r_vals = radial_profiles['radius'].values
fit_mask = (r_vals >= r_min) & (r_vals <= r_max)
r_fit = r_vals[fit_mask]

# Dimensioni del dataset
n_delays = radial_profiles.sizes['delay_time']
n_scans = radial_profiles.sizes['scan']

# Preparazione matrice risultati (j=delay, i=scan, 6 parametri)
# Parametri: [A, B(mu), C(sigma), D(slope), E(intercept), Area(A*C)]
fit_results = np.zeros((n_delays, n_scans, 6))

# --- 3. PREPARAZIONE FIGURA PLOTLY ---
ncols = 4
nrows = int(np.ceil(n_scans / ncols))
fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(n_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

# --- 4. DOPPIO CICLO: FIT E PLOT ---
print(f"Inizio il fit su {n_scans} scan e {n_delays} delay...")

for i in range(n_scans):
    for j in range(n_delays):
        # Estrazione dati per lo specifico scan e delay
        y_vals = radial_profiles.isel(delay_time=j, scan=i).values
        ydata = y_vals[fit_mask]
        
        # Stime iniziali e Limiti (Bounds)
        p0 = [np.max(ydata) - np.min(ydata), (r_min+r_max)/2, 3.0, 0.0, np.mean(ydata)]
        lower_bounds = [0, r_min, 0, -np.inf, -np.inf]
        upper_bounds = [np.inf, r_max, r_max-r_min, np.inf, np.inf]
        
        try:
            # Calcolo del fit
            popt, _ = curve_fit(gaussian_with_linear_bg, r_fit, ydata, 
                                p0=p0, bounds=(lower_bounds, upper_bounds), maxfev=2000)
            
            # Salvataggio nella matrice (Logica MATLAB)
            fit_results[j, i, 0] = popt[0]            # A
            fit_results[j, i, 1] = popt[1]            # B (Centro/mu)
            fit_results[j, i, 2] = popt[2]            # C (Larghezza/sigma)
            fit_results[j, i, 3] = popt[3]            # D (Pendenza fondo)
            fit_results[j, i, 4] = popt[4]            # E (Intercetta fondo)
            fit_results[j, i, 5] = popt[0] * popt[2]  # Area (A * C)
            
            # Se siamo nel delay scelto per il plot, aggiungiamo le curve al grafico
            if j == delay_to_plot:
                y_fit_curve = gaussian_with_linear_bg(r_fit, *popt)
                row = (i // ncols) + 1
                col = (i % ncols) + 1

                # Dati reali
                fig.add_trace(go.Scatter(x=r_fit, y=ydata, mode='lines+markers', 
                                         marker=dict(size=4), line=dict(width=1, color='royalblue'),
                                         showlegend=False), row=row, col=col)
                # Curva di fit
                fig.add_trace(go.Scatter(x=r_fit, y=y_fit_curve, mode='lines', 
                                         line=dict(dash='dash', width=2, color='red'),
                                         showlegend=False), row=row, col=col)
                
        except RuntimeError:
            # Se il fit fallisce, riempiamo con NaN
            fit_results[j, i, :] = np.nan

print("✅ Fit completato! Risultati salvati in 'fit_results'.")

# --- 5. VISUALIZZAZIONE DEL GRAFICO ---
fig.update_layout(
    height=250 * nrows, width=300 * ncols,
    title=f"Verifica Fit per delay_idx={delay_to_plot} (Raggio {r_min}-{r_max})",
    showlegend=False, hovermode="x unified"
)
fig.update_xaxes(title_text="Raggio (pixel)")
fig.update_yaxes(title_text="Intensità")
fig.show()

## Plots

## Amplitude

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# --- 1. SETTAGGI ---
n_cols = 4
# Prendiamo n_scans direttamente dal dataset se non è in memoria
n_scans = radial_profiles.sizes['scan']
n_rows = int(np.ceil(n_scans / n_cols))

# Estraiamo l'asse dei tempi
delay_times = radial_profiles['delay_time'].values

# --- 2. CREAZIONE FIGURA ---
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False # Lasciamo falso così ogni scan si adatta al proprio livello di segnale
)

# --- 3. PLOT DELLA DINAMICA PER OGNI SCAN ---
for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    # Estraiamo l'ampiezza (A) per tutti i delay dello scan 'j'
    # Ricorda: in fit_results l'indice 0 è l'Ampiezza (A)
    amplitudes = fit_results[:, j, 0]        

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes,
        mode='lines+markers',
        name=f"Scan {j}",
        marker=dict(size=5),
        line=dict(width=1.5),
        showlegend=False
    ), row=row, col=col)

# --- 4. LAYOUT FINALE ---
fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Dinamica dell'Ampiezza (A) vs Delay Time per ogni Scan",
    showlegend=False,
    hovermode="x unified"
)

fig.update_yaxes(title_text="Ampiezza (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# --- SETTAGGI PER QUESTO ESPERIMENTO ---
# ==========================================

# 1. Quale delay usiamo come riferimento "freddo"? (indice 0 = primo delay)
idx_baseline = 0 

# 2. Ci sono scan palesemente sballati che vuoi escludere dalla media finale?
bad_scans = [] 

# 3. Quale parametro di intensità vuoi guardare?
# 0 = Ampiezza pura (A)
# 5 = Area totale (A * sigma) ---> Spesso più stabile dell'ampiezza pura
parametro_scelto = 0

# ==========================================

# --- PREPARAZIONE DEI DATI ---
delays = radial_profiles['delay_time'].values
n_scans = radial_profiles.sizes['scan']

# Estraiamo l'intensità assoluta (Dati Puri)
dati_intensita = fit_results[:, :, parametro_scelto]

# Calcoliamo la normalizzazione (I / I_0) usando il delay scelto come baseline
intensita_t0 = dati_intensita[idx_baseline, :]
norm_intensita = dati_intensita / intensita_t0

# Identifichiamo gli scan buoni per fare la media
good_scans = [i for i in range(n_scans) if i not in bad_scans]

layout_template = dict(hovermode="x unified", xaxis_title="Delay Time", 
                       height=400, width=900, margin=dict(l=50, r=50, t=50, b=50))

# --- FIGURA 1: AMPIEZZA ASSOLUTA (Dati Puri) ---
fig1 = go.Figure()
for i in range(n_scans):
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    fig1.add_trace(go.Scatter(x=delays, y=dati_intensita[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig1.update_layout(**layout_template, title="1. Ampiezza Assoluta vs Delay (NON normalizzata)", yaxis_title="Intensità Assoluta")
fig1.show()


# --- FIGURA 2: INTENSITÀ NORMALIZZATA (Tutti gli scan) ---
fig2 = go.Figure()
for i in range(n_scans):
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    fig2.add_trace(go.Scatter(x=delays, y=norm_intensita[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig2.update_layout(**layout_template, title="2. Dinamica Intensità Normalizzata (I / I₀)", yaxis_title="Intensità Normalizzata (I / I₀)")
fig2.add_hline(y=1.0, line_dash="dash", line_color="black") # Linea di riferimento pre-pump
fig2.show()


# --- FIGURA 3: MEDIA FINALE (Solo scan buoni) ---
fig3 = go.Figure()

# 1. Estraiamo i dati puri degli scan buoni
dati_puri_good = dati_intensita[:, good_scans]

# 2. Facciamo la media dei dati PURI (cancella il rumore!)
mean_pura = np.nanmean(dati_puri_good, axis=1)

# 3. Normalizziamo la media usando i primi 3 punti (come fa il fit)
baseline_media = np.nanmean(mean_pura[:3])
mean_good_normalizzata = mean_pura / baseline_media

fig3.add_trace(go.Scatter(x=delays, y=mean_good_normalizzata, mode='lines+markers', 
                          line=dict(color='blue', width=3), 
                          name=f'Media Normalizzata ({len(good_scans)} scan)'))

fig3.update_layout(**layout_template, title=f"3. Media Intensità Reale (Esclusi scan: {bad_scans})", yaxis_title="Intensità Normalizzata (I / I₀)")
# Aggiungiamo la linea di riferimento all'1.0
fig3.add_hline(y=1.0, line_dash="dash", line_color="black")
fig3.show()

## Peak position

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# --- SETTAGGI PER QUESTO ESPERIMENTO ---
# ==========================================

# L'indice 0 è il primo delay (es. -200 fs).
idx_baseline = 0 

# 2. Ci sono scan palesemente sballati che vuoi escludere dalla media finale?
# Inserisci gli indici qui. Es: bad_scans = [0, 5, 19]. Lascia vuoto [] se sono tutti buoni.
bad_scans = [] 

# ==========================================

# --- PREPARAZIONE DEI DATI ---
delays = radial_profiles['delay_time'].values
n_scans = radial_profiles.sizes['scan']

# Estraiamo la Posizione del Centro (mu) 
positions = fit_results[:, :, 1]

# Calcoliamo la variazione % usando il delay scelto come baseline
pos_t0 = positions[idx_baseline, :]
delta_pos = ((positions - pos_t0) / pos_t0) * 100

# Identifichiamo gli scan buoni per fare la media
good_scans = [i for i in range(n_scans) if i not in bad_scans]

layout_template = dict(hovermode="x unified", xaxis_title="Delay Time", 
                       yaxis_title="Δμ / μ₀ (%)", height=450, width=900)

# --- FIGURA 1: Tutti gli scan (Ottima per fare diagnostica) ---
fig1 = go.Figure()
for i in range(n_scans):
    # Se lo scan è nella "lista nera", lo facciamo tratteggiato e più trasparente
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    
    fig1.add_trace(go.Scatter(x=delays, y=delta_pos[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig1.update_layout(**layout_template, title="1. Time evolution of peak center (each scan)")
fig1.add_hline(y=0, line_dash="dash", line_color="black")
fig1.show()


# --- FIGURA 2: Media FINALE---
fig2 = go.Figure()

# Estraiamo solo le colonne degli scan buoni e ne facciamo la media
delta_pos_good = delta_pos[:, good_scans]
mean_good = np.nanmean(delta_pos_good, axis=1)

fig2.add_trace(go.Scatter(x=delays, y=mean_good, mode='lines+markers', 
                          line=dict(color='red', width=3), 
                          name=f'Media ({len(good_scans)} scan)'))

fig2.update_layout(**layout_template, title=f"2. Average Time evolution of peak center")
fig2.add_hline(y=0, line_dash="dash", line_color="black")
fig2.show()

## Derivata

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import savgol_filter

# ==========================================
# --- PANNELLO DI CONTROLLO DIAGNOSTICA ---
# ==========================================
idx_delay = 0  # Usiamo il primo delay come riferimento
idx_scan = 0   # Usiamo il primo scan come riferimento

# Parametri di Smoothing da applicare ALLA DERIVATA
# window_length: quanti pixel usiamo per mediare (deve essere dispari, es. 5, 7, 9)
# polyorder: ordine del polinomio (2 è lo standard)
window_length = 5
polyorder = 2
# ==========================================

# 1. Estrazione del singolo profilo radiale dall'array globale
profile = radial_profiles.isel(delay_time=idx_delay, scan=idx_scan).values
r = np.arange(len(profile))

# 2. Operazioni Matematiche
# a) Calcolo della Derivata Prima nuda e cruda
derivative_raw = np.gradient(profile) 

# b) Smoothing applicato DOPO la derivata
derivative_sm = savgol_filter(derivative_raw, window_length, polyorder)

# 3. Plot Interattivo con Subplots (Assi X indipendenti!)
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=False,  # <--- MODIFICA CHIAVE QUI
    vertical_spacing=0.12, # Aumentato un po' per far spazio alle etichette X del grafico 1
    subplot_titles=(f"Derivata Prima Grezza (Delay {idx_delay}, Scan {idx_scan})", 
                    "Derivata Prima Smoothed (Filtro Savitzky-Golay)")
)

# Traccia 1: Derivata Prima Grezza (Pannello Superiore)
fig.add_trace(go.Scatter(x=r, y=derivative_raw, mode='lines+markers', name='Derivata Grezza', 
                         line=dict(color='gray', width=1), marker=dict(size=4)), row=1, col=1)

# Traccia 2: Derivata Prima Lisciata (Pannello Inferiore)
fig.add_trace(go.Scatter(x=r, y=derivative_sm, mode='lines', name='Derivata Smoothed', 
                         line=dict(color='red', width=3)), row=2, col=1)

# Linee dello zero per allineare l'occhio
fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="black", row=2, col=1)

fig.update_layout(
    height=800, width=900, 
    hovermode="x unified",
    template="plotly_white",
    title_text="Tool Diagnostico: Effetto dello Smoothing sulla Derivata"
)

# Etichette assi (Ora per entrambi i grafici)
fig.update_xaxes(title_text="Raggio (pixel)", row=1, col=1)
fig.update_xaxes(title_text="Raggio (pixel)", row=2, col=1)
fig.update_yaxes(title_text="Δ Intensità (Grezza)", row=1, col=1)
fig.update_yaxes(title_text="Δ Intensità (Smoothed)", row=2, col=1)

fig.show()

In [ ]:
import numpy as np
import xarray as xr
from scipy.signal import savgol_filter

# ==========================================
# --- PARAMETRI DERIVATA E SMOOTHING ---
# ==========================================
window_length = 5  # Stesso valore che hai scelto dal grafico diagnostico
polyorder = 2
# ==========================================

print("Inizio calcolo derivata e smoothing su tutto il dataset...")

# 1. Estraiamo la matrice 3D grezza (delay_time, scan, radius)
data_matrix = radial_profiles.values

# 2. Calcoliamo la derivata prima lungo l'asse del raggio (axis=-1, ovvero l'ultimo asse)
derivative_raw = np.gradient(data_matrix, axis=-1)

# 3. Applichiamo il filtro di Savitzky-Golay sempre lungo l'asse del raggio
derivative_smoothed = savgol_filter(derivative_raw, window_length, polyorder, axis=-1)

# 4. Ricreiamo un xarray DataArray per mantenere la comodità delle coordinate
radial_deriv_sm = xr.DataArray(
    derivative_smoothed,
    dims=radial_profiles.dims,      # Mantiene ('delay_time', 'scan', 'radius')
    coords=radial_profiles.coords,  # Mantiene i valori esatti di tempi, scan e pixel
    name="Smoothed_Derivative"
)

print("✅ Calcolo completato! Risultato salvato in 'radial_deriv_sm'.")

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ==========================================
# --- IMPOSTAZIONI AMPIEZZA (SU DATI FILTRATI) ---
# ==========================================
x1 = 63  # <--- Inserisci il limite sinistro
x2 = 80  # <--- Inserisci il limite destro

idx_baseline = 0
bad_scans = []
# ==========================================

n_delays = radial_deriv_sm.sizes['delay_time']
n_scans = radial_deriv_sm.sizes['scan']
delays = radial_deriv_sm['delay_time'].values

# PRENDIAMO I DATI GIÀ FILTRATI DAL TUO XARRAY
deriv_values_sm = radial_deriv_sm.values
peak_amplitude = np.zeros((n_delays, n_scans))

print("Calcolo Ampiezza (sui dati GIA' LISCIATI) in corso...")

for i in range(n_delays):
    for j in range(n_scans):
        # Prendiamo la derivata già lisciata
        ydata_sm = deriv_values_sm[i, j, :]
        
        # Estrai Ampiezza
        roi_y_sm = ydata_sm[x1:x2]
        peak_amplitude[i, j] = np.abs(np.max(roi_y_sm) - np.min(roi_y_sm))

# --- NORMALIZZAZIONE AMPIEZZA (I / I0) ---
good_scans = [s for s in range(n_scans) if s not in bad_scans]
amp_t0 = peak_amplitude[idx_baseline, :]
norm_amplitude = peak_amplitude / amp_t0
mean_amp = np.mean(norm_amplitude[:, good_scans], axis=1)

print("✅ Estrazione Ampiezza completata!")

# --- PLOT AMPIEZZA ---
fig_amp = go.Figure()
fig_amp.add_trace(go.Scatter(
    x=delays, y=mean_amp, mode='lines+markers', 
    line=dict(color='blue', width=3), name=f'Media ({len(good_scans)} scan)'
))
fig_amp.add_hline(y=1, line_dash="dash", line_color="black")

fig_amp.update_layout(
    title=f"Dinamica I/I₀ (Metodo Derivata Lisciata) - Picco [{x1}:{x2}]",
    xaxis_title="Delay Time (ps)", yaxis_title="I / I₀",
    hovermode="x unified", template="plotly_white", height=450, width=900
)
fig_amp.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ==========================================
# --- IMPOSTAZIONI CENTRO (SU DATI FILTRATI) ---
# ==========================================
x1 = 63  # <--- Limite sinistro
x2 = 80  # <--- Limite destro
q = 3    # <--- Finestra per il centro di massa (minimo ± q)

idx_baseline = 0
bad_scans = []
# ==========================================

n_delays = radial_deriv_sm.sizes['delay_time']
n_scans = radial_deriv_sm.sizes['scan']
delays = radial_deriv_sm['delay_time'].values

deriv_values_sm = radial_deriv_sm.values  
peak_position = np.zeros((n_delays, n_scans))
for i in range(n_delays):
    for j in range(n_scans):
        # Prendiamo la derivata già lisciata
        ydata_sm = deriv_values_sm[i, j, :]
        
        # 1. Trova il minimo locale
        roi_y_sm = ydata_sm[x1:x2]
        local_min_idx = np.argmin(roi_y_sm)
        xmin = x1 + local_min_idx
        
        # 2. Finestra ± q
        xvect = np.arange(xmin - q, xmin + q + 1)
        window_y_sm = ydata_sm[xmin - q : xmin + q + 1]
        
        # 3. Centro di massa
        peak_position[i, j] = np.sum(xvect * window_y_sm) / np.sum(window_y_sm)

# --- NORMALIZZAZIONE POSIZIONE (Δμ / μ0 in %) ---
good_scans = [s for s in range(n_scans) if s not in bad_scans]
pos_t0 = peak_position[idx_baseline, :]
delta_pos = ((peak_position - pos_t0) / pos_t0) * 100
mean_pos = np.mean(delta_pos[:, good_scans], axis=1)

print("✅ Estrazione Posizione completata!")

# --- PLOT POSIZIONE ---
fig_pos = go.Figure()
fig_pos.add_trace(go.Scatter(
    x=delays, y=mean_pos, mode='lines+markers', 
    line=dict(color='red', width=3), name=f'Media ({len(good_scans)} scan)'
))
fig_pos.add_hline(y=0, line_dash="dash", line_color="black")

fig_pos.update_layout(
    title=f"Evoluzione Δμ/μ₀ (Centro di Massa Derivata Lisciata, q={q})",
    xaxis_title="Delay Time (ps)", yaxis_title="Δμ / μ₀ (%)",
    hovermode="x unified", template="plotly_white", height=450, width=900
)
fig_pos.show()

## SECOND PEAK

Single gaussian fit

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. DEFINIZIONE FUNZIONE ---
def gaussian_with_linear_bg(r, A, mu, sigma, slope, intercept):
    return A * np.exp(-(r - mu)**2 / (2 * sigma**2)) + slope * r + intercept

# --- 2. SETTAGGI DEL RANGE E DEL DELAY DA PLOTTARE ---
r_min = 89  # Sostituisci con 140 se vuoi il picco 5 del tuo script MATLAB
r_max = 104   # Sostituisci con 155 se vuoi il picco 5 del tuo script MATLAB

delay_to_plot = 0  # L'indice del delay che vuoi ispezionare visivamente

# Estrazione asse x (raggio) e maschera
r_vals = radial_profiles['radius'].values
fit_mask = (r_vals >= r_min) & (r_vals <= r_max)
r_fit = r_vals[fit_mask]

# Dimensioni del dataset
n_delays = radial_profiles.sizes['delay_time']
n_scans = radial_profiles.sizes['scan']

# Preparazione matrice risultati (j=delay, i=scan, 6 parametri)
# Parametri: [A, B(mu), C(sigma), D(slope), E(intercept), Area(A*C)]
fit_results_second = np.zeros((n_delays, n_scans, 6))

# --- 3. PREPARAZIONE FIGURA PLOTLY ---
ncols = 4
nrows = int(np.ceil(n_scans / ncols))
fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(n_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

# --- 4. DOPPIO CICLO: FIT E PLOT ---
print(f"Inizio il fit su {n_scans} scan e {n_delays} delay...")

for i in range(n_scans):
    for j in range(n_delays):
        # Estrazione dati per lo specifico scan e delay
        y_vals = radial_profiles.isel(delay_time=j, scan=i).values
        ydata = y_vals[fit_mask]
        
        # Stime iniziali e Limiti (Bounds)
        p0 = [np.max(ydata) - np.min(ydata), (r_min+r_max)/2, 3.0, 0.0, np.mean(ydata)]
        lower_bounds = [0, r_min, 0, -np.inf, -np.inf]
        upper_bounds = [np.inf, r_max, r_max-r_min, np.inf, np.inf]
        
        try:
            # Calcolo del fit
            popt, _ = curve_fit(gaussian_with_linear_bg, r_fit, ydata, 
                                p0=p0, bounds=(lower_bounds, upper_bounds), maxfev=2000)
            
            # Salvataggio nella matrice (Logica MATLAB)
            fit_results_second[j, i, 0] = popt[0]            # A
            fit_results_second[j, i, 1] = popt[1]            # B (Centro/mu)
            fit_results_second[j, i, 2] = popt[2]            # C (Larghezza/sigma)
            fit_results_second[j, i, 3] = popt[3]            # D (Pendenza fondo)
            fit_results_second[j, i, 4] = popt[4]            # E (Intercetta fondo)
            fit_results_second[j, i, 5] = popt[0] * popt[2]  # Area (A * C)
            
            # Se siamo nel delay scelto per il plot, aggiungiamo le curve al grafico
            if j == delay_to_plot:
                y_fit_curve = gaussian_with_linear_bg(r_fit, *popt)
                row = (i // ncols) + 1
                col = (i % ncols) + 1

                # Dati reali
                fig.add_trace(go.Scatter(x=r_fit, y=ydata, mode='lines+markers', 
                                         marker=dict(size=4), line=dict(width=1, color='royalblue'),
                                         showlegend=False), row=row, col=col)
                # Curva di fit
                fig.add_trace(go.Scatter(x=r_fit, y=y_fit_curve, mode='lines', 
                                         line=dict(dash='dash', width=2, color='red'),
                                         showlegend=False), row=row, col=col)
                
        except RuntimeError:
            # Se il fit fallisce, riempiamo con NaN
            fit_results_second[j, i, :] = np.nan

print("✅ Fit completato! Risultati salvati in 'fit_results'.")

# --- 5. VISUALIZZAZIONE DEL GRAFICO ---
fig.update_layout(
    height=250 * nrows, width=300 * ncols,
    title=f"Verifica Fit per delay_idx={delay_to_plot} (Raggio {r_min}-{r_max})",
    showlegend=False, hovermode="x unified"
)
fig.update_xaxes(title_text="Raggio (pixel)")
fig.update_yaxes(title_text="Intensità")
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# --- 1. SETTAGGI ---
n_cols = 4
# Prendiamo n_scans direttamente dal dataset se non è in memoria
n_scans = radial_profiles.sizes['scan']
n_rows = int(np.ceil(n_scans / n_cols))

# Estraiamo l'asse dei tempi
delay_times = radial_profiles['delay_time'].values

# --- 2. CREAZIONE FIGURA ---
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False # Lasciamo falso così ogni scan si adatta al proprio livello di segnale
)

# --- 3. PLOT DELLA DINAMICA PER OGNI SCAN ---
for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    # Estraiamo l'ampiezza (A) per tutti i delay dello scan 'j'
    # Ricorda: in fit_results l'indice 0 è l'Ampiezza (A)
    amplitudes = fit_results_second[:, j, 0]        

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes,
        mode='lines+markers',
        name=f"Scan {j}",
        marker=dict(size=5),
        line=dict(width=1.5),
        showlegend=False
    ), row=row, col=col)

# --- 4. LAYOUT FINALE ---
fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Dinamica dell'Ampiezza (A) vs Delay Time per ogni Scan",
    showlegend=False,
    hovermode="x unified"
)

fig.update_yaxes(title_text="Ampiezza (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# --- SETTAGGI PER QUESTO ESPERIMENTO ---
# ==========================================

# 1. Quale delay usiamo come riferimento "freddo"? (indice 0 = primo delay)
idx_baseline = 0 

# 2. Ci sono scan palesemente sballati che vuoi escludere dalla media finale?
bad_scans = [] 

# 3. Quale parametro di intensità vuoi guardare?
# 0 = Ampiezza pura (A)
# 5 = Area totale (A * sigma) ---> Spesso più stabile dell'ampiezza pura
parametro_scelto = 0

# ==========================================

# --- PREPARAZIONE DEI DATI ---
delays = radial_profiles['delay_time'].values
n_scans = radial_profiles.sizes['scan']

# Estraiamo l'intensità assoluta (Dati Puri)
dati_intensita = fit_results_second[:, :, parametro_scelto]

# Calcoliamo la normalizzazione (I / I_0) usando il delay scelto come baseline
intensita_t0 = dati_intensita[idx_baseline, :]
norm_intensita = dati_intensita / intensita_t0

# Identifichiamo gli scan buoni per fare la media
good_scans = [i for i in range(n_scans) if i not in bad_scans]

layout_template = dict(hovermode="x unified", xaxis_title="Delay Time", 
                       height=400, width=900, margin=dict(l=50, r=50, t=50, b=50))

# --- FIGURA 1: AMPIEZZA ASSOLUTA (Dati Puri) ---
fig1 = go.Figure()
for i in range(n_scans):
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    fig1.add_trace(go.Scatter(x=delays, y=dati_intensita[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig1.update_layout(**layout_template, title="1. Ampiezza Assoluta vs Delay (NON normalizzata)", yaxis_title="Intensità Assoluta")
fig1.show()


# --- FIGURA 2: INTENSITÀ NORMALIZZATA (Tutti gli scan) ---
fig2 = go.Figure()
for i in range(n_scans):
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    fig2.add_trace(go.Scatter(x=delays, y=norm_intensita[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig2.update_layout(**layout_template, title="2. Dinamica Intensità Normalizzata (I / I₀)", yaxis_title="Intensità Normalizzata (I / I₀)")
fig2.add_hline(y=1.0, line_dash="dash", line_color="black") # Linea di riferimento pre-pump
fig2.show()


# --- FIGURA 3: MEDIA FINALE (Solo scan buoni) ---
fig3 = go.Figure()

# 1. Estraiamo i dati puri degli scan buoni
dati_puri_good = dati_intensita[:, good_scans]

# 2. Facciamo la media dei dati PURI (cancella il rumore!)
mean_pura = np.nanmean(dati_puri_good, axis=1)

# 3. Normalizziamo la media usando i primi 3 punti (come fa il fit)
baseline_media = np.nanmean(mean_pura[:3])
mean_good_normalizzata = mean_pura / baseline_media

fig3.add_trace(go.Scatter(x=delays, y=mean_good_normalizzata, mode='lines+markers', 
                          line=dict(color='blue', width=3), 
                          name=f'Media Normalizzata ({len(good_scans)} scan)'))

fig3.update_layout(**layout_template, title=f"3. Media Intensità Reale (Esclusi scan: {bad_scans})", yaxis_title="Intensità Normalizzata (I / I₀)")

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# --- SETTAGGI PER QUESTO ESPERIMENTO ---
# ==========================================

# L'indice 0 è il primo delay (es. -200 fs).
idx_baseline = 0 

# 2. Ci sono scan palesemente sballati che vuoi escludere dalla media finale?
# Inserisci gli indici qui. Es: bad_scans = [0, 5, 19]. Lascia vuoto [] se sono tutti buoni.
bad_scans = [0] 

# ==========================================

# --- PREPARAZIONE DEI DATI ---
delays = radial_profiles['delay_time'].values
n_scans = radial_profiles.sizes['scan']

# Estraiamo la Posizione del Centro (mu) 
positions = fit_results_second[:, :, 1]

# Calcoliamo la variazione % usando il delay scelto come baseline
pos_t0 = positions[idx_baseline, :]
delta_pos = ((positions - pos_t0) / pos_t0) * 100

# Identifichiamo gli scan buoni per fare la media
good_scans = [i for i in range(n_scans) if i not in bad_scans]

layout_template = dict(hovermode="x unified", xaxis_title="Delay Time", 
                       yaxis_title="Δμ / μ₀ (%)", height=450, width=900)



# --- FIGURA 2: Media FINALE---
fig2 = go.Figure()

# Estraiamo solo le colonne degli scan buoni e ne facciamo la media
delta_pos_good = delta_pos[:, good_scans]
mean_good = np.nanmean(delta_pos_good, axis=1)

fig2.add_trace(go.Scatter(x=delays, y=mean_good, mode='lines+markers', 
                          line=dict(color='red', width=3), 
                          name=f'Media ({len(good_scans)} scan)'))

fig2.update_layout(**layout_template, title=f"2. Average Time evolution of peak center")
fig2.add_hline(y=0, line_dash="dash", line_color="black")
fig2.show()

## Derivata

In [ ]:
import numpy as np
import plotly.graph_objects as go

x1_p2 = 88  
x2_p2 = 104
idx_baseline = 0
bad_scans = []
# ==========================================

n_delays = radial_deriv_sm.sizes['delay_time']
n_scans = radial_deriv_sm.sizes['scan']
delays = radial_deriv_sm['delay_time'].values

# Preleviamo sempre dal cubo filtrato globale
deriv_values_sm = radial_deriv_sm.values  
peak_amplitude_p2 = np.zeros((n_delays, n_scans))

print("Calcolo Ampiezza (Picco 2) in corso...")

for i in range(n_delays):
    for j in range(n_scans):
        ydata_sm = deriv_values_sm[i, j, :]
        roi_y_sm = ydata_sm[x1_p2:x2_p2]
        
        peak_amplitude_p2[i, j] = np.abs(np.max(roi_y_sm) - np.min(roi_y_sm))

# --- NORMALIZZAZIONE AMPIEZZA PICCO 2 ---
good_scans = [s for s in range(n_scans) if s not in bad_scans]
amp_t0_p2 = peak_amplitude_p2[idx_baseline, :]
norm_amplitude_p2 = peak_amplitude_p2 / amp_t0_p2
mean_amp_p2 = np.mean(norm_amplitude_p2[:, good_scans], axis=1)

print("✅ Estrazione Ampiezza Picco 2 completata! Array: 'mean_amp_p2'")

# --- PLOT AMPIEZZA PICCO 2 ---
fig_amp_p2 = go.Figure()
fig_amp_p2.add_trace(go.Scatter(
    x=delays, y=mean_amp_p2, mode='lines+markers', 
    line=dict(color='green', width=3), name=f'Media Picco 2'
))
fig_amp_p2.add_hline(y=1, line_dash="dash", line_color="black")

fig_amp_p2.update_layout(
    title=f"Dinamica I/I₀ (Picco 2) - Intervallo [{x1_p2}:{x2_p2}]",
    xaxis_title="Delay Time (ps)", yaxis_title="I / I₀",
    hovermode="x unified", template="plotly_white", height=450, width=900
)
fig_amp_p2.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ==========================================
# --- IMPOSTAZIONI CENTRO (PICCO 2) ---
# ==========================================
# Usa gli stessi limiti che hai messo sopra per il Picco 2
x1_p2 = 88  
x2_p2 = 104
q_p2 = 3   

idx_baseline = 0
bad_scans = []
# ==========================================

n_delays = radial_deriv_sm.sizes['delay_time']
n_scans = radial_deriv_sm.sizes['scan']

deriv_values_sm = radial_deriv_sm.values  
peak_position_p2 = np.zeros((n_delays, n_scans))

print("Calcolo Posizione (Picco 2) in corso...")

for i in range(n_delays):
    for j in range(n_scans):
        ydata_sm = deriv_values_sm[i, j, :]
        
        roi_y_sm = ydata_sm[x1_p2:x2_p2]
        local_min_idx = np.argmin(roi_y_sm)
        xmin = x1_p2 + local_min_idx
        
        xvect = np.arange(xmin - q_p2, xmin + q_p2 + 1)
        window_y_sm = ydata_sm[xmin - q_p2 : xmin + q_p2 + 1]
        
        peak_position_p2[i, j] = np.sum(xvect * window_y_sm) / np.sum(window_y_sm)

# --- NORMALIZZAZIONE POSIZIONE PICCO 2 ---
good_scans = [s for s in range(n_scans) if s not in bad_scans]
pos_t0_p2 = peak_position_p2[idx_baseline, :]
delta_pos_p2 = ((peak_position_p2 - pos_t0_p2) / pos_t0_p2) * 100
mean_pos_p2 = np.mean(delta_pos_p2[:, good_scans], axis=1)

print("✅ Estrazione Posizione Picco 2 completata! Array: 'mean_pos_p2'")

# --- PLOT POSIZIONE PICCO 2 ---
fig_pos_p2 = go.Figure()
fig_pos_p2.add_trace(go.Scatter(
    x=delays, y=mean_pos_p2, mode='lines+markers', 
    line=dict(color='orange', width=3), name=f'Media Picco 2'
))
fig_pos_p2.add_hline(y=0, line_dash="dash", line_color="black")

fig_pos_p2.update_layout(
    title=f"Evoluzione Δμ/μ₀ (Picco 2) - Intervallo [{x1_p2}:{x2_p2}]",
    xaxis_title="Delay Time (ps)", yaxis_title="Δμ / μ₀ (%)",
    hovermode="x unified", template="plotly_white", height=450, width=900
)
fig_pos_p2.show()

# Third peak

Single gaussian

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. DEFINIZIONE FUNZIONE ---
def gaussian_with_linear_bg(r, A, mu, sigma, slope, intercept):
    return A * np.exp(-(r - mu)**2 / (2 * sigma**2)) + slope * r + intercept

# --- 2. SETTAGGI DEL RANGE E DEL DELAY DA PLOTTARE ---
r_min = 105  # Inizio del Picco 3
r_max = 127  # Fine del Picco 3

delay_to_plot = 0  # L'indice del delay che vuoi ispezionare visivamente

# Estrazione asse x (raggio) e maschera
r_vals = radial_profiles['radius'].values
fit_mask = (r_vals >= r_min) & (r_vals <= r_max)
r_fit = r_vals[fit_mask]

# Dimensioni del dataset
n_delays = radial_profiles.sizes['delay_time']
n_scans = radial_profiles.sizes['scan']

# --- NUOVA MATRICE PER IL PICCO 3 ---
# Parametri: [A, B(mu), C(sigma), D(slope), E(intercept), Area(A*C)]
fit_results_third = np.zeros((n_delays, n_scans, 6))

# --- 3. PREPARAZIONE FIGURA PLOTLY ---
ncols = 4
nrows = int(np.ceil(n_scans / ncols))
fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(n_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

# --- 4. DOPPIO CICLO: FIT E PLOT ---
print(f"Inizio il fit su {n_scans} scan e {n_delays} delay per il PICCO 3...")

for i in range(n_scans):
    for j in range(n_delays):
        # Estrazione dati per lo specifico scan e delay
        y_vals = radial_profiles.isel(delay_time=j, scan=i).values
        ydata = y_vals[fit_mask]
        
        # Stime iniziali e Limiti (Bounds)
        p0 = [np.max(ydata) - np.min(ydata), (r_min+r_max)/2, 3.0, 0.0, np.mean(ydata)]
        lower_bounds = [0, r_min, 0, -np.inf, -np.inf]
        upper_bounds = [np.inf, r_max, r_max-r_min, np.inf, np.inf]
        
        try:
            # Calcolo del fit
            popt, _ = curve_fit(gaussian_with_linear_bg, r_fit, ydata, 
                                p0=p0, bounds=(lower_bounds, upper_bounds), maxfev=2000)
            
            # Salvataggio nella NUOVA matrice (Logica MATLAB)
            fit_results_third[j, i, 0] = popt[0]            # A
            fit_results_third[j, i, 1] = popt[1]            # B (Centro/mu)
            fit_results_third[j, i, 2] = popt[2]            # C (Larghezza/sigma)
            fit_results_third[j, i, 3] = popt[3]            # D (Pendenza fondo)
            fit_results_third[j, i, 4] = popt[4]            # E (Intercetta fondo)
            fit_results_third[j, i, 5] = popt[0] * popt[2]  # Area (A * C)
            
            # Se siamo nel delay scelto per il plot, aggiungiamo le curve al grafico
            if j == delay_to_plot:
                y_fit_curve = gaussian_with_linear_bg(r_fit, *popt)
                row = (i // ncols) + 1
                col = (i % ncols) + 1

                # Dati reali
                fig.add_trace(go.Scatter(x=r_fit, y=ydata, mode='lines+markers', 
                                         marker=dict(size=4), line=dict(width=1, color='royalblue'),
                                         showlegend=False), row=row, col=col)
                # Curva di fit
                fig.add_trace(go.Scatter(x=r_fit, y=y_fit_curve, mode='lines', 
                                         line=dict(dash='dash', width=2, color='red'),
                                         showlegend=False), row=row, col=col)
                
        except RuntimeError:
            # Se il fit fallisce, riempiamo con NaN
            fit_results_third[j, i, :] = np.nan

print("✅ Fit del Picco 3 completato! Risultati salvati in 'fit_results_third'.")

# --- 5. VISUALIZZAZIONE DEL GRAFICO ---
fig.update_layout(
    height=250 * nrows, width=300 * ncols,
    title=f"Verifica Fit Picco 3 per delay_idx={delay_to_plot} (Raggio {r_min}-{r_max})",
    showlegend=False, hovermode="x unified"
)
fig.update_xaxes(title_text="Raggio (pixel)")
fig.update_yaxes(title_text="Intensità")
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# --- 1. SETTAGGI ---
n_cols = 4
# Prendiamo n_scans direttamente dal dataset se non è in memoria
n_scans = radial_profiles.sizes['scan']
n_rows = int(np.ceil(n_scans / n_cols))

# Estraiamo l'asse dei tempi
delay_times = radial_profiles['delay_time'].values

# --- 2. CREAZIONE FIGURA ---
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False # Lasciamo falso così ogni scan si adatta al proprio livello di segnale
)

# --- 3. PLOT DELLA DINAMICA PER OGNI SCAN ---
for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    # Estraiamo l'ampiezza (A) per tutti i delay dello scan 'j'
    # ATTENZIONE: Qui usiamo la NUOVA matrice fit_results_third!
    amplitudes = fit_results_third[:, j, 0]        

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes,
        mode='lines+markers',
        name=f"Scan {j}",
        marker=dict(size=5),
        line=dict(width=1.5),
        showlegend=False
    ), row=row, col=col)

# --- 4. LAYOUT FINALE ---
fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Dinamica dell'Ampiezza (A) vs Delay Time per ogni Scan - PICCO 3",
    showlegend=False,
    hovermode="x unified"
)

fig.update_yaxes(title_text="Ampiezza (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# --- SETTAGGI PER QUESTO ESPERIMENTO ---
# ==========================================

# 1. Quale delay usiamo come riferimento "freddo"? (indice 0 = primo delay)
idx_baseline = 0 

# 2. Ci sono scan palesemente sballati che vuoi escludere dalla media finale?
bad_scans = [] 

# 3. Quale parametro di intensità vuoi guardare?
# 0 = Ampiezza pura (A)
# 5 = Area totale (A * sigma) ---> Spesso più stabile dell'ampiezza pura
parametro_scelto = 0

# ==========================================

# --- PREPARAZIONE DEI DATI ---
delays = radial_profiles['delay_time'].values
n_scans = radial_profiles.sizes['scan']

# Estraiamo l'intensità assoluta (Dati Puri)
dati_intensita = fit_results_third[:, :, parametro_scelto]

# Calcoliamo la normalizzazione (I / I_0) usando il delay scelto come baseline
intensita_t0 = dati_intensita[idx_baseline, :]
norm_intensita = dati_intensita / intensita_t0

# Identifichiamo gli scan buoni per fare la media
good_scans = [i for i in range(n_scans) if i not in bad_scans]

layout_template = dict(hovermode="x unified", xaxis_title="Delay Time", 
                       height=400, width=900, margin=dict(l=50, r=50, t=50, b=50))

# --- FIGURA 1: AMPIEZZA ASSOLUTA (Dati Puri) ---
fig1 = go.Figure()
for i in range(n_scans):
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    fig1.add_trace(go.Scatter(x=delays, y=dati_intensita[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig1.update_layout(**layout_template, title="1. Ampiezza Assoluta vs Delay (NON normalizzata)", yaxis_title="Intensità Assoluta")
fig1.show()


# --- FIGURA 2: INTENSITÀ NORMALIZZATA (Tutti gli scan) ---
fig2 = go.Figure()
for i in range(n_scans):
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    fig2.add_trace(go.Scatter(x=delays, y=norm_intensita[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig2.update_layout(**layout_template, title="2. Dinamica Intensità Normalizzata (I / I₀)", yaxis_title="Intensità Normalizzata (I / I₀)")
fig2.add_hline(y=1.0, line_dash="dash", line_color="black") # Linea di riferimento pre-pump
fig2.show()


# --- FIGURA 3: MEDIA FINALE (Solo scan buoni) ---
fig3 = go.Figure()

# 1. Estraiamo i dati puri degli scan buoni
dati_puri_good = dati_intensita[:, good_scans]

# 2. Facciamo la media dei dati PURI (cancella il rumore!)
mean_pura = np.nanmean(dati_puri_good, axis=1)

# 3. Normalizziamo la media usando i primi 3 punti (come fa il fit)
baseline_media = np.nanmean(mean_pura[:3])
mean_good_normalizzata = mean_pura / baseline_media

fig3.add_trace(go.Scatter(x=delays, y=mean_good_normalizzata, mode='lines+markers', 
                          line=dict(color='blue', width=3), 
                          name=f'Media Normalizzata ({len(good_scans)} scan)'))

fig3.update_layout(**layout_template, title=f"3. Media Intensità Reale (Esclusi scan: {bad_scans})", yaxis_title="Intensità Normalizzata (I / I₀)")

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# --- SETTAGGI PER QUESTO ESPERIMENTO ---
# ==========================================

# L'indice 0 è il primo delay (es. -200 fs).
idx_baseline = 0 

# 2. Ci sono scan palesemente sballati che vuoi escludere dalla media finale?
# Inserisci gli indici qui. Es: bad_scans = [0, 5, 19]. Lascia vuoto [] se sono tutti buoni.
bad_scans = [] 

# ==========================================

# --- PREPARAZIONE DEI DATI ---
delays = radial_profiles['delay_time'].values
n_scans = radial_profiles.sizes['scan']

# Estraiamo la Posizione del Centro (mu) 
positions = fit_results_third[:, :, 1]

# Calcoliamo la variazione % usando il delay scelto come baseline
pos_t0 = positions[idx_baseline, :]
delta_pos = ((positions - pos_t0) / pos_t0) * 100

# Identifichiamo gli scan buoni per fare la media
good_scans = [i for i in range(n_scans) if i not in bad_scans]

layout_template = dict(hovermode="x unified", xaxis_title="Delay Time", 
                       yaxis_title="Δμ / μ₀ (%)", height=450, width=900)

# --- FIGURA 1: Tutti gli scan (Ottima per fare diagnostica) ---
fig1 = go.Figure()
for i in range(n_scans):
    # Se lo scan è nella "lista nera", lo facciamo tratteggiato e più trasparente
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    
    fig1.add_trace(go.Scatter(x=delays, y=delta_pos[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig1.update_layout(**layout_template, title="1. Time evolution of peak center (each scan)")
fig1.add_hline(y=0, line_dash="dash", line_color="black")
fig1.show()


# --- FIGURA 2: Media FINALE---
fig2 = go.Figure()

# Estraiamo solo le colonne degli scan buoni e ne facciamo la media
delta_pos_good = delta_pos[:, good_scans]
mean_good = np.nanmean(delta_pos_good, axis=1)

fig2.add_trace(go.Scatter(x=delays, y=mean_good, mode='lines+markers', 
                          line=dict(color='red', width=3), 
                          name=f'Media ({len(good_scans)} scan)'))

fig2.update_layout(**layout_template, title=f"2. Average Time evolution of peak center (Esclusi scan: {bad_scans})")
fig2.add_hline(y=0, line_dash="dash", line_color="black")
fig2.show()

# Fourth peak

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. DEFINIZIONE FUNZIONE ---
def gaussian_with_linear_bg(r, A, mu, sigma, slope, intercept):
    return A * np.exp(-(r - mu)**2 / (2 * sigma**2)) + slope * r + intercept

# --- 2. SETTAGGI DEL RANGE E DEL DELAY DA PLOTTARE ---
r_min = 134  # Inizio del Picco 3
r_max = 149  # Fine del Picco 3

delay_to_plot = 0  # L'indice del delay che vuoi ispezionare visivamente

# Estrazione asse x (raggio) e maschera
r_vals = radial_profiles['radius'].values
fit_mask = (r_vals >= r_min) & (r_vals <= r_max)
r_fit = r_vals[fit_mask]

# Dimensioni del dataset
n_delays = radial_profiles.sizes['delay_time']
n_scans = radial_profiles.sizes['scan']

# --- NUOVA MATRICE PER IL PICCO 4 ---
# Parametri: [A, B(mu), C(sigma), D(slope), E(intercept), Area(A*C)]
fit_results_fourth = np.zeros((n_delays, n_scans, 6))

# --- 3. PREPARAZIONE FIGURA PLOTLY ---
ncols = 4
nrows = int(np.ceil(n_scans / ncols))
fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(n_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

# --- 4. DOPPIO CICLO: FIT E PLOT ---
print(f"Inizio il fit su {n_scans} scan e {n_delays} delay per il PICCO 4...")

for i in range(n_scans):
    for j in range(n_delays):
        # Estrazione dati per lo specifico scan e delay
        y_vals = radial_profiles.isel(delay_time=j, scan=i).values
        ydata = y_vals[fit_mask]
        
        # Stime iniziali e Limiti (Bounds)
        p0 = [np.max(ydata) - np.min(ydata), (r_min+r_max)/2, 3.0, 0.0, np.mean(ydata)]
        lower_bounds = [0, r_min, 0, -np.inf, -np.inf]
        upper_bounds = [np.inf, r_max, r_max-r_min, np.inf, np.inf]
        
        try:
            # Calcolo del fit
            popt, _ = curve_fit(gaussian_with_linear_bg, r_fit, ydata, 
                                p0=p0, bounds=(lower_bounds, upper_bounds), maxfev=2000)
            
            # Salvataggio nella NUOVA matrice (Logica MATLAB)
            fit_results_fourth[j, i, 0] = popt[0]            # A
            fit_results_fourth[j, i, 1] = popt[1]            # B (Centro/mu)
            fit_results_fourth[j, i, 2] = popt[2]            # C (Larghezza/sigma)
            fit_results_fourth[j, i, 3] = popt[3]            # D (Pendenza fondo)
            fit_results_fourth[j, i, 4] = popt[4]            # E (Intercetta fondo)
            fit_results_fourth[j, i, 5] = popt[0] * popt[2]  # Area (A * C)
            
            # Se siamo nel delay scelto per il plot, aggiungiamo le curve al grafico
            if j == delay_to_plot:
                y_fit_curve = gaussian_with_linear_bg(r_fit, *popt)
                row = (i // ncols) + 1
                col = (i % ncols) + 1

                # Dati reali
                fig.add_trace(go.Scatter(x=r_fit, y=ydata, mode='lines+markers', 
                                         marker=dict(size=4), line=dict(width=1, color='royalblue'),
                                         showlegend=False), row=row, col=col)
                # Curva di fit
                fig.add_trace(go.Scatter(x=r_fit, y=y_fit_curve, mode='lines', 
                                         line=dict(dash='dash', width=2, color='red'),
                                         showlegend=False), row=row, col=col)
                
        except RuntimeError:
            # Se il fit fallisce, riempiamo con NaN
            fit_results_fourth[j, i, :] = np.nan

print("✅ Fit del Picco 4 completato! Risultati salvati in 'fit_results_fourth'.")

# --- 5. VISUALIZZAZIONE DEL GRAFICO ---
fig.update_layout(
    height=250 * nrows, width=300 * ncols,
    title=f"Verifica Fit Picco 4 per delay_idx={delay_to_plot} (Raggio {r_min}-{r_max})",
    showlegend=False, hovermode="x unified"
)
fig.update_xaxes(title_text="Raggio (pixel)")
fig.update_yaxes(title_text="Intensità")
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# --- 1. SETTAGGI ---
n_cols = 4
# Prendiamo n_scans direttamente dal dataset se non è in memoria
n_scans = radial_profiles.sizes['scan']
n_rows = int(np.ceil(n_scans / n_cols))

# Estraiamo l'asse dei tempi
delay_times = radial_profiles['delay_time'].values

# --- 2. CREAZIONE FIGURA ---
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False # Lasciamo falso così ogni scan si adatta al proprio livello di segnale
)

# --- 3. PLOT DELLA DINAMICA PER OGNI SCAN ---
for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    # Estraiamo l'ampiezza (A) per tutti i delay dello scan 'j'
    # ATTENZIONE: Qui usiamo la NUOVA matrice fit_results_fourth!
    amplitudes = fit_results_fourth[:, j, 0]        

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes,
        mode='lines+markers',
        name=f"Scan {j}",
        marker=dict(size=5),
        line=dict(width=1.5),
        showlegend=False
    ), row=row, col=col)

# --- 4. LAYOUT FINALE ---
fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Dinamica dell'Ampiezza (A) vs Delay Time per ogni Scan - PICCO 3",
    showlegend=False,
    hovermode="x unified"
)

fig.update_yaxes(title_text="Ampiezza (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# --- SETTAGGI PER QUESTO ESPERIMENTO ---
# ==========================================

# 1. Quale delay usiamo come riferimento "freddo"? (indice 0 = primo delay)
idx_baseline = 0 

# 2. Ci sono scan palesemente sballati che vuoi escludere dalla media finale?
bad_scans = [] 

# 3. Quale parametro di intensità vuoi guardare?
# 0 = Ampiezza pura (A)
# 5 = Area totale (A * sigma) ---> Spesso più stabile dell'ampiezza pura
parametro_scelto = 0

# ==========================================

# --- PREPARAZIONE DEI DATI ---
delays = radial_profiles['delay_time'].values
n_scans = radial_profiles.sizes['scan']

# Estraiamo l'intensità assoluta (Dati Puri)
dati_intensita = fit_results_fourth[:, :, parametro_scelto]

# Calcoliamo la normalizzazione (I / I_0) usando il delay scelto come baseline
intensita_t0 = dati_intensita[idx_baseline, :]
norm_intensita = dati_intensita / intensita_t0

# Identifichiamo gli scan buoni per fare la media
good_scans = [i for i in range(n_scans) if i not in bad_scans]

layout_template = dict(hovermode="x unified", xaxis_title="Delay Time", 
                       height=400, width=900, margin=dict(l=50, r=50, t=50, b=50))

# --- FIGURA 1: AMPIEZZA ASSOLUTA (Dati Puri) ---
fig1 = go.Figure()
for i in range(n_scans):
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    fig1.add_trace(go.Scatter(x=delays, y=dati_intensita[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig1.update_layout(**layout_template, title="1. Ampiezza Assoluta vs Delay (NON normalizzata)", yaxis_title="Intensità Assoluta")
fig1.show()


# --- FIGURA 2: INTENSITÀ NORMALIZZATA (Tutti gli scan) ---
fig2 = go.Figure()
for i in range(n_scans):
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    fig2.add_trace(go.Scatter(x=delays, y=norm_intensita[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig2.update_layout(**layout_template, title="2. Dinamica Intensità Normalizzata (I / I₀)", yaxis_title="Intensità Normalizzata (I / I₀)")
fig2.add_hline(y=1.0, line_dash="dash", line_color="black") # Linea di riferimento pre-pump
fig2.show()


# --- FIGURA 3: MEDIA FINALE (Solo scan buoni) ---
fig3 = go.Figure()

# 1. Estraiamo i dati puri degli scan buoni
dati_puri_good = dati_intensita[:, good_scans]

# 2. Facciamo la media dei dati PURI (cancella il rumore!)
mean_pura = np.nanmean(dati_puri_good, axis=1)

# 3. Normalizziamo la media usando i primi 3 punti (come fa il fit)
baseline_media = np.nanmean(mean_pura[:3])
mean_good_normalizzata = mean_pura / baseline_media

fig3.add_trace(go.Scatter(x=delays, y=mean_good_normalizzata, mode='lines+markers', 
                          line=dict(color='blue', width=3), 
                          name=f'Media Normalizzata ({len(good_scans)} scan)'))

fig3.update_layout(**layout_template, title=f"3. Media Intensità Reale (Esclusi scan: {bad_scans})", yaxis_title="Intensità Normalizzata (I / I₀)")

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# --- SETTAGGI PER QUESTO ESPERIMENTO ---
# ==========================================

# L'indice 0 è il primo delay (es. -200 fs).
idx_baseline = 0 

# 2. Ci sono scan palesemente sballati che vuoi escludere dalla media finale?
# Inserisci gli indici qui. Es: bad_scans = [0, 5, 19]. Lascia vuoto [] se sono tutti buoni.
bad_scans = [] 

# ==========================================

# --- PREPARAZIONE DEI DATI ---
delays = radial_profiles['delay_time'].values
n_scans = radial_profiles.sizes['scan']

# Estraiamo la Posizione del Centro (mu) 
positions = fit_results_fourth[:, :, 1]

# Calcoliamo la variazione % usando il delay scelto come baseline
pos_t0 = positions[idx_baseline, :]
delta_pos = ((positions - pos_t0) / pos_t0) * 100

# Identifichiamo gli scan buoni per fare la media
good_scans = [i for i in range(n_scans) if i not in bad_scans]

layout_template = dict(hovermode="x unified", xaxis_title="Delay Time", 
                       yaxis_title="Δμ / μ₀ (%)", height=450, width=900)

# --- FIGURA 1: Tutti gli scan (Ottima per fare diagnostica) ---
fig1 = go.Figure()
for i in range(n_scans):
    # Se lo scan è nella "lista nera", lo facciamo tratteggiato e più trasparente
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    
    fig1.add_trace(go.Scatter(x=delays, y=delta_pos[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig1.update_layout(**layout_template, title="1. Time evolution of peak center (each scan)")
fig1.add_hline(y=0, line_dash="dash", line_color="black")
fig1.show()


# --- FIGURA 2: Media FINALE---
fig2 = go.Figure()

# Estraiamo solo le colonne degli scan buoni e ne facciamo la media
delta_pos_good = delta_pos[:, good_scans]
mean_good = np.nanmean(delta_pos_good, axis=1)

fig2.add_trace(go.Scatter(x=delays, y=mean_good, mode='lines+markers', 
                          line=dict(color='red', width=3), 
                          name=f'Media ({len(good_scans)} scan)'))

fig2.update_layout(**layout_template, title=f"2. Average Time evolution of peak center (Esclusi scan: {bad_scans})")
fig2.add_hline(y=0, line_dash="dash", line_color="black")
fig2.show()

# Fifth peak

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. DEFINIZIONE FUNZIONE ---
def gaussian_with_linear_bg(r, A, mu, sigma, slope, intercept):
    return A * np.exp(-(r - mu)**2 / (2 * sigma**2)) + slope * r + intercept

# --- 2. SETTAGGI DEL RANGE E DEL DELAY DA PLOTTARE ---
r_min = 174  # Inizio del Picco 5
r_max = 196  # Fine del Picco 5

delay_to_plot = 0  # L'indice del delay che vuoi ispezionare visivamente

# Estrazione asse x (raggio) e maschera
r_vals = radial_profiles['radius'].values
fit_mask = (r_vals >= r_min) & (r_vals <= r_max)
r_fit = r_vals[fit_mask]

# Dimensioni del dataset
n_delays = radial_profiles.sizes['delay_time']
n_scans = radial_profiles.sizes['scan']

# --- NUOVA MATRICE PER IL PICCO 5  ---
# Parametri: [A, B(mu), C(sigma), D(slope), E(intercept), Area(A*C)]
fit_results_fifth = np.zeros((n_delays, n_scans, 6))

# --- 3. PREPARAZIONE FIGURA PLOTLY ---
ncols = 4
nrows = int(np.ceil(n_scans / ncols))
fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(n_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

# --- 4. DOPPIO CICLO: FIT E PLOT ---
print(f"Inizio il fit su {n_scans} scan e {n_delays} delay per il PICCO 5...")

for i in range(n_scans):
    for j in range(n_delays):
        # Estrazione dati per lo specifico scan e delay
        y_vals = radial_profiles.isel(delay_time=j, scan=i).values
        ydata = y_vals[fit_mask]
        
        # Stime iniziali e Limiti (Bounds)
        p0 = [np.max(ydata) - np.min(ydata), (r_min+r_max)/2, 3.0, 0.0, np.mean(ydata)]
        lower_bounds = [0, r_min, 0, -np.inf, -np.inf]
        upper_bounds = [np.inf, r_max, r_max-r_min, np.inf, np.inf]
        
        try:
            # Calcolo del fit
            popt, _ = curve_fit(gaussian_with_linear_bg, r_fit, ydata, 
                                p0=p0, bounds=(lower_bounds, upper_bounds), maxfev=2000)
            
            # Salvataggio nella NUOVA matrice (Logica MATLAB)
            fit_results_fifth[j, i, 0] = popt[0]            # A
            fit_results_fifth[j, i, 1] = popt[1]            # B (Centro/mu)
            fit_results_fifth[j, i, 2] = popt[2]            # C (Larghezza/sigma)
            fit_results_fifth[j, i, 3] = popt[3]            # D (Pendenza fondo)
            fit_results_fifth[j, i, 4] = popt[4]            # E (Intercetta fondo)
            fit_results_fifth[j, i, 5] = popt[0] * popt[2]  # Area (A * C)
            
            # Se siamo nel delay scelto per il plot, aggiungiamo le curve al grafico
            if j == delay_to_plot:
                y_fit_curve = gaussian_with_linear_bg(r_fit, *popt)
                row = (i // ncols) + 1
                col = (i % ncols) + 1

                # Dati reali
                fig.add_trace(go.Scatter(x=r_fit, y=ydata, mode='lines+markers', 
                                         marker=dict(size=4), line=dict(width=1, color='royalblue'),
                                         showlegend=False), row=row, col=col)
                # Curva di fit
                fig.add_trace(go.Scatter(x=r_fit, y=y_fit_curve, mode='lines', 
                                         line=dict(dash='dash', width=2, color='red'),
                                         showlegend=False), row=row, col=col)
                
        except RuntimeError:
            # Se il fit fallisce, riempiamo con NaN
            fit_results_fifth[j, i, :] = np.nan

print("✅ Fit del Picco 5 completato! Risultati salvati in 'fit_results_fifth'.")

# --- 5. VISUALIZZAZIONE DEL GRAFICO ---
fig.update_layout(
    height=250 * nrows, width=300 * ncols,
    title=f"Verifica Fit Picco 5 per delay_idx={delay_to_plot} (Raggio {r_min}-{r_max})",
    showlegend=False, hovermode="x unified"
)
fig.update_xaxes(title_text="Raggio (pixel)")
fig.update_yaxes(title_text="Intensità")
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# --- 1. SETTAGGI ---
n_cols = 4
# Prendiamo n_scans direttamente dal dataset se non è in memoria
n_scans = radial_profiles.sizes['scan']
n_rows = int(np.ceil(n_scans / n_cols))

# Estraiamo l'asse dei tempi
delay_times = radial_profiles['delay_time'].values

# --- 2. CREAZIONE FIGURA ---
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False # Lasciamo falso così ogni scan si adatta al proprio livello di segnale
)

# --- 3. PLOT DELLA DINAMICA PER OGNI SCAN ---
for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    # Estraiamo l'ampiezza (A) per tutti i delay dello scan 'j'
    # ATTENZIONE: Qui usiamo la NUOVA matrice fit_results_fifth!
    amplitudes = fit_results_fifth[:, j, 0]        

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes,
        mode='lines+markers',
        name=f"Scan {j}",
        marker=dict(size=5),
        line=dict(width=1.5),
        showlegend=False
    ), row=row, col=col)

# --- 4. LAYOUT FINALE ---
fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Dinamica dell'Ampiezza (A) vs Delay Time per ogni Scan - PICCO 3",
    showlegend=False,
    hovermode="x unified"
)

fig.update_yaxes(title_text="Ampiezza (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# --- SETTAGGI PER QUESTO ESPERIMENTO ---
# ==========================================

# 1. Quale delay usiamo come riferimento "freddo"? (indice 0 = primo delay)
idx_baseline = 0 

# 2. Ci sono scan palesemente sballati che vuoi escludere dalla media finale?
bad_scans = [16] 

# 3. Quale parametro di intensità vuoi guardare?
# 0 = Ampiezza pura (A)
# 5 = Area totale (A * sigma) ---> Spesso più stabile dell'ampiezza pura
parametro_scelto = 0

# ==========================================

# --- PREPARAZIONE DEI DATI ---
delays = radial_profiles['delay_time'].values
n_scans = radial_profiles.sizes['scan']

# Estraiamo l'intensità assoluta (Dati Puri)
dati_intensita = fit_results_fifth[:, :, parametro_scelto]

# Calcoliamo la normalizzazione (I / I_0) usando il delay scelto come baseline
intensita_t0 = dati_intensita[idx_baseline, :]
norm_intensita = dati_intensita / intensita_t0

# Identifichiamo gli scan buoni per fare la media
good_scans = [i for i in range(n_scans) if i not in bad_scans]

layout_template = dict(hovermode="x unified", xaxis_title="Delay Time", 
                       height=400, width=900, margin=dict(l=50, r=50, t=50, b=50))

# --- FIGURA 1: AMPIEZZA ASSOLUTA (Dati Puri) ---
fig1 = go.Figure()
for i in range(n_scans):
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    fig1.add_trace(go.Scatter(x=delays, y=dati_intensita[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig1.update_layout(**layout_template, title="1. Ampiezza Assoluta vs Delay (NON normalizzata)", yaxis_title="Intensità Assoluta")
fig1.show()


# --- FIGURA 2: INTENSITÀ NORMALIZZATA (Tutti gli scan) ---
fig2 = go.Figure()
for i in range(n_scans):
    line_style = dict(dash='dot', width=1) if i in bad_scans else dict(width=2)
    opacity = 0.3 if i in bad_scans else 0.7
    fig2.add_trace(go.Scatter(x=delays, y=norm_intensita[:, i], mode='lines', 
                              name=f'Scan {i}', opacity=opacity, line=line_style))

fig2.update_layout(**layout_template, title="2. Dinamica Intensità Normalizzata (I / I₀)", yaxis_title="Intensità Normalizzata (I / I₀)")
fig2.add_hline(y=1.0, line_dash="dash", line_color="black") # Linea di riferimento pre-pump
fig2.show()


# --- FIGURA 3: MEDIA FINALE (Solo scan buoni) ---
fig3 = go.Figure()

# 1. Estraiamo i dati puri degli scan buoni
dati_puri_good = dati_intensita[:, good_scans]

# 2. Facciamo la media dei dati PURI (cancella il rumore!)
mean_pura = np.nanmean(dati_puri_good, axis=1)

# 3. Normalizziamo la media usando i primi 3 punti (come fa il fit)
baseline_media = np.nanmean(mean_pura[:3])
mean_good_normalizzata = mean_pura / baseline_media

fig3.add_trace(go.Scatter(x=delays, y=mean_good_normalizzata, mode='lines+markers', 
                          line=dict(color='blue', width=3), 
                          name=f'Media Normalizzata ({len(good_scans)} scan)'))

fig3.update_layout(**layout_template, title=f"3. Media Intensità Reale (Esclusi scan: {bad_scans})", yaxis_title="Intensità Normalizzata (I / I₀)")

## Fit of average amplitude curves

Single exponential

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# --- PANNELLO DI CONTROLLO ---
# ==========================================
# 1 = Primo picco (es. a 72 pixel, dati in 'fit_results')
# 2 = Secondo picco (es. a 95 pixel, dati in 'fit_results_second')
picco_scelto = 2

# 0 = Ampiezza pura (A)
# 5 = Area totale (A * sigma) 
parametro_scelto = 0

# Eventuali scan da scartare
bad_scans = []
# ==========================================

# --- 1. SELEZIONE DEI DATI ---
# Scegliamo la matrice giusta in base al picco scelto
if picco_scelto == 1:
    dati_raw = fit_results
    nome_picco = "Peak 1"
if picco_scelto == 2:
    dati_raw = fit_results_second 
    nome_picco = "Peak 2"
if picco_scelto == 3:
    dati_raw = fit_results_third 
    nome_picco = "Peak 3"

delays_times = radial_profiles['delay_time'].values
good_scans = [i for i in range(radial_profiles.sizes['scan']) if i not in bad_scans]

# Estraiamo i dati, facciamo la media degli scan buoni
intensita_estratti = dati_raw[:, good_scans, parametro_scelto]
A2_values = np.nanmean(intensita_estratti, axis=1)

# Rimuoviamo eventuali NaN
valid_idx = ~np.isnan(A2_values)
times_clean = delays_times[valid_idx]
A2_clean = A2_values[valid_idx]

# --- 2. FIT ESPONENZIALE SINGOLO ---
# Normalization factor, first 3 values
norm_factor = np.mean(A2_clean[:3])
A2_normalized = A2_clean / norm_factor

def step_exponential(t, A, tau, t0):
    dt = np.maximum(t - t0, 0) # Previene crash matematici
    return 1 - np.heaviside(t - t0, 1) * A * (1 - np.exp(-dt / tau))

# Stime iniziali
A_guess = 1.0
tau_guess = 10.0
# abs() nel gradiente ci assicura di trovare il salto sia se l'intensità sale sia se scende
t0_guess = times_clean[np.argmax(np.abs(np.gradient(A2_normalized)))]  

p0 = [A_guess, tau_guess, t0_guess]

# Limiti [A, tau, t0]. A può essere negativo se la curva sale invece di scendere!
bounds = ([-2.0, 1e-3, times_clean.min()], [2.0, np.inf, times_clean.max()])

try:
    popt, pcov = curve_fit(step_exponential, times_clean, A2_normalized, p0=p0, bounds=bounds)
    A_fit, tau_fit, t0_fit = popt
    
    # --- CALCOLO INCERTEZZE ---
    perr = np.sqrt(np.diag(pcov))
    A_err, tau_err, t0_err = perr
    
    print(f"--- RISULTATI FIT SINGOLO ({nome_picco}) ---")
    print(f"t0  = {t0_fit:.3f} ± {t0_err:.3f} ps")
    print(f"A   = {A_fit:.3f} ± {A_err:.3f}")
    print(f"tau = {tau_fit:.3f} ± {tau_err:.3f} ps")

    t_fit_hires = np.linspace(times_clean.min(), times_clean.max(), 500)
    fitted_curve = step_exponential(t_fit_hires, *popt)

except RuntimeError as e:
    print(f"Fit fallito: {e}")
    t_fit_hires = times_clean
    fitted_curve = np.full_like(times_clean, np.nan)

# --- 3. GRAFICO PLOTLY ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=times_clean, y=A2_normalized, mode='lines+markers',
    marker=dict(size=8), line=dict(width=2), name=f'Data ({nome_picco})'
))

fig.add_trace(go.Scatter(
    x=t_fit_hires, y=fitted_curve, mode='lines',
    name='Single Exp Fit', line=dict(color='red', width=3)
))

fig.update_layout(
    title=f'Time Evolution of Normalized Amplitude - {nome_picco}',
    xaxis_title='Delay Time', yaxis_title='Normalized Amplitude',
    hovermode='closest', template='plotly_white', width=900, height=500
)
fig.show()

Double exponential

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# --- PANNELLO DI CONTROLLO ---
# ==========================================
# 1 = Primo picco (es. a 72 pixel, dati in 'fit_results')
# 2 = Secondo picco (es. a 95 pixel, dati in 'fit_results_peak2')
picco_scelto = 2

parametro_scelto = 0
bad_scans = []
# ==========================================

# --- 1. SELEZIONE DEI DATI ---
if picco_scelto == 1:
    dati_raw = fit_results
    nome_picco = "Peak 1"
else:
    dati_raw = fit_results_second
    nome_picco = "Peak 2"

delays_times = radial_profiles['delay_time'].values
good_scans = [i for i in range(radial_profiles.sizes['scan']) if i not in bad_scans]

intensita_estratti = dati_raw[:, good_scans, parametro_scelto]
A2_values = np.nanmean(intensita_estratti, axis=1)

valid_idx = ~np.isnan(A2_values)
times_clean = delays_times[valid_idx]
A2_clean = A2_values[valid_idx]

# --- 2. FIT BI-ESPONENZIALE ---
norm_factor = np.mean(A2_clean[:3])
A2_normalized = A2_clean / norm_factor

def step_biexponential(t, A1, tau1, A2, tau2, t0):
    dt = np.maximum(t - t0, 0)
    comp1 = A1 * (1 - np.exp(-dt / tau1))
    comp2 = A2 * (1 - np.exp(-dt / tau2))
    return 1 - np.heaviside(t - t0, 1) * (comp1 + comp2)

# Stime iniziali
A1_guess, tau1_guess = 0.5, 2.0
A2_guess, tau2_guess = 0.5, 50.0
t0_guess = times_clean[np.argmax(np.abs(np.gradient(A2_normalized)))]  

p0 = [A1_guess, tau1_guess, A2_guess, tau2_guess, t0_guess]

# Limiti [A1, tau1, A2, tau2, t0]
bounds = ([-2.0, 1e-3, -2.0, 1e-3, times_clean.min()], 
          [ 2.0, np.inf, 2.0, np.inf, times_clean.max()])

try:
    popt, pcov = curve_fit(step_biexponential, times_clean, A2_normalized, p0=p0, bounds=bounds)
    A1_fit, tau1_fit, A2_fit, tau2_fit, t0_fit = popt
    
    # --- CALCOLO INCERTEZZE ---
    perr = np.sqrt(np.diag(pcov))
    A1_err, tau1_err, A2_err, tau2_err, t0_err = perr
    
    print(f"--- RISULTATI FIT BI-ESPONENZIALE ({nome_picco}) ---")
    print(f"t0   = {t0_fit:.3f} ± {t0_err:.3f} ps")
    print(f"A1   = {A1_fit:.3f} ± {A1_err:.3f}, tau1 = {tau1_fit:.3f} ± {tau1_err:.3f} ps")
    print(f"A2   = {A2_fit:.3f} ± {A2_err:.3f}, tau2 = {tau2_fit:.3f} ± {tau2_err:.3f} ps")

    t_fit_hires = np.linspace(times_clean.min(), times_clean.max(), 500)
    fitted_curve = step_biexponential(t_fit_hires, *popt)

except RuntimeError as e:
    print(f"Fit fallito: {e}")
    t_fit_hires = times_clean
    fitted_curve = np.full_like(times_clean, np.nan)

# --- 3. GRAFICO PLOTLY ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=times_clean, y=A2_normalized, mode='lines+markers',
    marker=dict(size=8), line=dict(width=2), name=f'Data ({nome_picco})'
))

fig.add_trace(go.Scatter(
    x=t_fit_hires, y=fitted_curve, mode='lines',
    name='Bi-Exp Fit', line=dict(color='red', width=3)
))

fig.update_layout(
    title=f'Time Evolution of Normalized Amplitude (Bi-Exp) - {nome_picco}',
    xaxis_title='Delay Time', yaxis_title='Normalized Amplitude',
    hovermode='closest', template='plotly_white', width=900, height=500
)
fig.show()

Remove the last two experimental points

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# --- PANNELLO DI CONTROLLO ---
# ==========================================
picco_scelto = 1
parametro_scelto = 0
bad_scans = []
# ==========================================

# --- 1. SELEZIONE DEI DATI ---
if picco_scelto == 1:
    dati_raw = fit_results
    nome_picco = "Peak 1"
if picco_scelto == 2:
    dati_raw = fit_results_second 
    nome_picco = "Peak 2"
if picco_scelto == 3:
    dati_raw = fit_results_third 
    nome_picco = "Peak 3"

delays_times = radial_profiles['delay_time'].values
good_scans = [i for i in range(radial_profiles.sizes['scan']) if i not in bad_scans]

intensita_estratti = dati_raw[:, good_scans, parametro_scelto]
A2_values = np.nanmean(intensita_estratti, axis=1)

# Rimuoviamo eventuali NaN
valid_idx = ~np.isnan(A2_values)
times_clean = delays_times[valid_idx]
A2_clean = A2_values[valid_idx]

# --- 🎯 NUOVO: FILTRO TEMPORALE (TAGLIA LA CODA) ---
# Manteniamo SOLO i tempi strettamente minori di -50 ps
time_mask = times_clean < -50
times_clean = times_clean[time_mask]
A2_clean = A2_clean[time_mask]
# --------------------------------------------------

# --- 2. FIT ESPONENZIALE SINGOLO ---
norm_factor = np.mean(A2_clean[:3])
A2_normalized = A2_clean / norm_factor

def step_exponential(t, A, tau, t0):
    dt = np.maximum(t - t0, 0) 
    return 1 - np.heaviside(t - t0, 1) * A * (1 - np.exp(-dt / tau))

A_guess = 1.0
tau_guess = 10.0
t0_guess = times_clean[np.argmax(np.abs(np.gradient(A2_normalized)))]  

p0 = [A_guess, tau_guess, t0_guess]
bounds = ([-2.0, 1e-3, times_clean.min()], [2.0, np.inf, times_clean.max()])

try:
    popt, pcov = curve_fit(step_exponential, times_clean, A2_normalized, p0=p0, bounds=bounds)
    A_fit, tau_fit, t0_fit = popt
    
    perr = np.sqrt(np.diag(pcov))
    A_err, tau_err, t0_err = perr
    
    print(f"--- RISULTATI FIT SINGOLO ({nome_picco} - Troncato) ---")
    print(f"t0  = {t0_fit:.3f} ± {t0_err:.3f} ps")
    print(f"A   = {A_fit:.3f} ± {A_err:.3f}")
    print(f"tau = {tau_fit:.3f} ± {tau_err:.3f} ps")

    # Facciamo arrivare la linea del fit fino a -50 per vederla bene nel grafico
    t_fit_hires = np.linspace(times_clean.min(), -50, 500)
    fitted_curve = step_exponential(t_fit_hires, *popt)

except RuntimeError as e:
    print(f"Fit fallito: {e}")
    t_fit_hires = times_clean
    fitted_curve = np.full_like(times_clean, np.nan)

# --- 3. GRAFICO PLOTLY ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=times_clean, y=A2_normalized, mode='lines+markers',
    marker=dict(size=8), line=dict(width=2), name=f'Data ({nome_picco})'
))

fig.add_trace(go.Scatter(
    x=t_fit_hires, y=fitted_curve, mode='lines',
    name='Single Exp Fit', line=dict(color='red', width=3)
))

fig.update_layout(
    title=f'Time Evolution of Normalized Amplitude - {nome_picco} (Tagliati i tempi >= -50ps)',
    xaxis_title='Delay Time', yaxis_title='Normalized Amplitude',
    hovermode='closest', template='plotly_white', width=900, height=500
)
fig.show()

Rescaling

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# --- PANNELLO DI CONTROLLO ---
# ==========================================
picco_scelto = 1

# 0 = Ampiezza pura (A)
# 5 = Area totale (A * sigma) 
parametro_scelto = 0

# Eventuali scan da scartare
bad_scans = []

# --- NOVITÀ: RISCALAMENTO ASSE Y ---
# Fissiamo l'asse Y per poter confrontare visivamente grafici diversi.
# Usa [0.85, 1.02] per farci stare dentro comodamente il salto grande dell'Oro (che arriva a ~0.87)
usa_asse_y_fisso = True
range_asse_y = [0.85, 1.02] 
# ==========================================

# --- 1. SELEZIONE DEI DATI ---
# Scegliamo la matrice giusta in base al picco scelto
if picco_scelto == 1:
    dati_raw = fit_results
    nome_picco = "Peak 1"
if picco_scelto == 2:
    dati_raw = fit_results_second 
    nome_picco = "Peak 2"
if picco_scelto == 3:
    dati_raw = fit_results_third 
    nome_picco = "Peak 3"

delays_times = radial_profiles['delay_time'].values
good_scans = [i for i in range(radial_profiles.sizes['scan']) if i not in bad_scans]

# Estraiamo i dati, facciamo la media degli scan buoni
intensita_estratti = dati_raw[:, good_scans, parametro_scelto]
A2_values = np.nanmean(intensita_estratti, axis=1)

# Rimuoviamo eventuali NaN
valid_idx = ~np.isnan(A2_values)
times_clean = delays_times[valid_idx]
A2_clean = A2_values[valid_idx]

# --- 2. FIT ESPONENZIALE SINGOLO ---
# Normalization factor, first 3 values
norm_factor = np.mean(A2_clean[:3])
A2_normalized = A2_clean / norm_factor

def step_exponential(t, A, tau, t0):
    dt = np.maximum(t - t0, 0) # Previene crash matematici
    return 1 - np.heaviside(t - t0, 1) * A * (1 - np.exp(-dt / tau))

# Stime iniziali
A_guess = 1.0
tau_guess = 10.0
# abs() nel gradiente ci assicura di trovare il salto sia se l'intensità sale sia se scende
t0_guess = times_clean[np.argmax(np.abs(np.gradient(A2_normalized)))]  

p0 = [A_guess, tau_guess, t0_guess]

# Limiti [A, tau, t0]. A può essere negativo se la curva sale invece di scendere!
bounds = ([-2.0, 1e-3, times_clean.min()], [2.0, np.inf, times_clean.max()])

try:
    popt, pcov = curve_fit(step_exponential, times_clean, A2_normalized, p0=p0, bounds=bounds)
    A_fit, tau_fit, t0_fit = popt
    
    # --- CALCOLO INCERTEZZE ---
    perr = np.sqrt(np.diag(pcov))
    A_err, tau_err, t0_err = perr
    
    print(f"--- RISULTATI FIT SINGOLO ({nome_picco}) ---")
    print(f"t0  = {t0_fit:.3f} ± {t0_err:.3f} ps")
    print(f"A   = {A_fit:.3f} ± {A_err:.3f}")
    print(f"tau = {tau_fit:.3f} ± {tau_err:.3f} ps")

    t_fit_hires = np.linspace(times_clean.min(), times_clean.max(), 500)
    fitted_curve = step_exponential(t_fit_hires, *popt)

except RuntimeError as e:
    print(f"Fit fallito: {e}")
    t_fit_hires = times_clean
    fitted_curve = np.full_like(times_clean, np.nan)

# --- 3. GRAFICO PLOTLY ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=times_clean, y=A2_normalized, mode='lines+markers',
    marker=dict(size=8), line=dict(width=2), name=f'Data ({nome_picco})'
))

fig.add_trace(go.Scatter(
    x=t_fit_hires, y=fitted_curve, mode='lines',
    name='Single Exp Fit', line=dict(color='red', width=3)
))

# Aggiorniamo il layout con il range dell'asse Y se richiesto
layout_update = dict(
    title=f'Time Evolution of Normalized Amplitude - {nome_picco}',
    xaxis_title='Delay Time (ps)', 
    yaxis_title='Normalized Amplitude',
    hovermode='closest', 
    template='plotly_white', 
    width=900, 
    height=500
)

# Applica il range fisso se la variabile è True
if usa_asse_y_fisso:
    layout_update['yaxis'] = dict(range=range_asse_y)

fig.update_layout(**layout_update)
fig.show()